In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, when

DATA_PATH = "data/raw/prosperLoanData.csv"

spark = SparkSession.builder \
    .appName("Check Prosper Loan Dataset") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("\n========== DATASET VALIDATION ==========")

if not os.path.exists(DATA_PATH):
    print(f"File not found: {DATA_PATH}")
    spark.stop()
    exit()

df = spark.read.csv(DATA_PATH, header=True, inferSchema=True)

row_count = df.count()
column_count = len(df.columns)

print(f"Dataset name: Prosper Loan Dataset")
print(f"Source: Kaggle")
print(f"Number of rows: {row_count}")
print(f"Number of columns: {column_count}")
print(f"Rows > 100000: {row_count > 100000}")
print(f"Columns > 10: {column_count > 10}")

print("\n========== SELECTED SCHEMA ==========")
selected_columns = [
    "ListingKey",
    "LoanStatus",
    "Term",
    "BorrowerAPR",
    "BorrowerRate",
    "ProsperScore",
    "CreditScoreRangeLower",
    "CreditScoreRangeUpper",
    "DebtToIncomeRatio",
    "IncomeRange",
    "EmploymentStatus",
    "Occupation",
    "LoanOriginalAmount",
    "MonthlyLoanPayment",
    "Investors"
]

df.select(selected_columns).printSchema()

print("\n========== SAMPLE DATA ==========")
df.select(selected_columns).show(5, truncate=False)

print("\n========== TOP 15 MISSING VALUE COLUMNS ==========")
missing_row = df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).collect()[0].asDict()

missing_list = sorted(
    missing_row.items(),
    key=lambda x: x[1],
    reverse=True
)

for column, missing_count in missing_list[:15]:
    missing_percent = (missing_count / row_count) * 100
    print(f"{column}: {missing_count} missing values ({missing_percent:.2f}%)")

print("\n========== CONCLUSION ==========")
if row_count > 100000 and column_count > 10:
    print("The dataset satisfies the Big Data project requirement.")
else:
    print("The dataset does not satisfy the Big Data project requirement.")

spark.stop()


========== DATASET VALIDATION ==========
Dataset name: Prosper Loan Dataset
Source: Kaggle
Number of rows: 113937
Number of columns: 81
Rows > 100000: True
Columns > 10: True

========== SELECTED SCHEMA ==========
root
 |-- ListingKey: string (nullable = true)
 |-- LoanStatus: string (nullable = true)
 |-- Term: integer (nullable = true)
 |-- BorrowerAPR: double (nullable = true)
 |-- BorrowerRate: double (nullable = true)
 |-- ProsperScore: double (nullable = true)
 |-- CreditScoreRangeLower: integer (nullable = true)
 |-- CreditScoreRangeUpper: integer (nullable = true)
 |-- DebtToIncomeRatio: double (nullable = true)
 |-- IncomeRange: string (nullable = true)
 |-- EmploymentStatus: string (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- LoanOriginalAmount: integer (nullable = true)
 |-- MonthlyLoanPayment: double (nullable = true)
 |-- Investors: integer (nullable = true)


========== SAMPLE DATA ==========
+-----------------------+----------+----+-----------+------

In [7]:
from pyspark.sql import SparkSession

HDFS_PATH = "hdfs://localhost:9000/bigdata/prosper_loan/raw/prosperLoanData.csv"

spark = SparkSession.builder \
    .appName("Read Prosper Loan From HDFS") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("\n========== READ DATA FROM HDFS ==========")

print(f"HDFS Path: {HDFS_PATH}")

df = spark.read.csv(
    HDFS_PATH,
    header=True,
    inferSchema=True
)

row_count = df.count()
column_count = len(df.columns)

print("\n========== DATASET SUMMARY ==========")
print(f"Number of rows: {row_count}")
print(f"Number of columns: {column_count}")

print("\n========== SELECTED SCHEMA ==========")
df.printSchema()

print("\n========== SAMPLE DATA FROM HDFS ==========")
df.select(
    "LoanStatus",
    "BorrowerAPR",
    "ProsperScore",
    "CreditScoreRangeLower",
    "DebtToIncomeRatio",
    "IncomeRange",
    "Occupation",
    "EmploymentStatus",
    "LoanOriginalAmount",
    "MonthlyLoanPayment"
).show(10, truncate=False)

print("\n========== CONCLUSION ==========")
print("Spark successfully read Prosper Loan Dataset directly from HDFS.")

spark.stop()


========== READ DATA FROM HDFS ==========
HDFS Path: hdfs://localhost:9000/bigdata/prosper_loan/raw/prosperLoanData.csv

========== DATASET SUMMARY ==========
Number of rows: 113937
Number of columns: 81

========== SELECTED SCHEMA ==========
root
 |-- ListingKey: string (nullable = true)
 |-- ListingNumber: integer (nullable = true)
 |-- ListingCreationDate: timestamp (nullable = true)
 |-- CreditGrade: string (nullable = true)
 |-- Term: integer (nullable = true)
 |-- LoanStatus: string (nullable = true)
 |-- ClosedDate: timestamp (nullable = true)
 |-- BorrowerAPR: double (nullable = true)
 |-- BorrowerRate: double (nullable = true)
 |-- LenderYield: double (nullable = true)
 |-- EstimatedEffectiveYield: double (nullable = true)
 |-- EstimatedLoss: double (nullable = true)
 |-- EstimatedReturn: double (nullable = true)
 |-- ProsperRating (numeric): integer (nullable = true)
 |-- ProsperRating (Alpha): string (nullable = true)
 |-- ProsperScore: double (nullable = true)
 |-- Listing

In [8]:
from pyspark.sql import SparkSession


HDFS_INPUT_PATH = "hdfs://localhost:9000/bigdata/prosper_loan/raw/prosperLoanData.csv"
HDFS_OUTPUT_PATH = "hdfs://localhost:9000/bigdata/prosper_loan/processed/prosper_loan_reduced"

spark = (
    SparkSession.builder
    .appName("Prosper Loan Domain Feature Reduction")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

feature_progress = []


def print_status(step_name, dataframe):
    print(f"\n========== {step_name} ==========")
    print(f"Rows: {dataframe.count()}")
    print(f"Columns: {len(dataframe.columns)}")


def print_feature_list(title, columns):
    print(f"{title} ({len(columns)}):")
    if not columns:
        print("- None")
        return

    for column_name in columns:
        print(f"- {column_name}")


def record_feature_progress(step_name, dataframe):
    feature_progress.append((step_name, len(dataframe.columns)))


def print_initial_domain_filtering_summary(columns_before, columns_removed, columns_remaining):
    print("\n========== INITIAL DOMAIN FILTERING SUMMARY ==========")
    print(f"Columns before filtering: {columns_before}")
    print(f"Columns removed: {columns_removed}")
    print(f"Columns remaining: {columns_remaining}")

    print("\nRemoved column categories:")
    print("- Identifier columns")
    print("- Duplicate loan identifiers")
    print("- Internal platform tracking fields")
    print("- Historical payment/recovery accounting fields")
    print("- Post-loan outcome leakage variables")

    print("\nReason:")
    print(
        "These columns are not suitable as predictive features because they either "
        "uniquely identify records, contain duplicated identifiers, represent internal "
        "platform bookkeeping fields, or leak information that would not be available "
        "at loan origination time."
    )

    print("\nConclusion:")
    print(
        f"Initial domain filtering reduced the dataset from {columns_before} to "
        f"{columns_remaining} features."
    )
    print(
        "The removed attributes do not contribute meaningful predictive information "
        "for loan risk modeling and may introduce noise or data leakage."
    )
    print(
        "The remaining features will be evaluated further through domain-based grouping "
        "and feature analysis."
    )


def print_feature_reduction_progress():
    print("\n========== FEATURE REDUCTION PROGRESS ==========")
    print(f"{'Step':<45} Columns")
    print("-" * 55)
    for step_name, column_count in feature_progress:
        print(f"{step_name:<45} {column_count}")


def safe_drop(dataframe, columns_to_drop):
    existing_columns = [col_name for col_name in columns_to_drop if col_name in dataframe.columns]
    missing_columns = [col_name for col_name in columns_to_drop if col_name not in dataframe.columns]

    if missing_columns:
        print_feature_list("Missing columns skipped", missing_columns)

    if not existing_columns:
        return dataframe

    print_feature_list("Dropped columns", existing_columns)
    return dataframe.drop(*existing_columns)


print("\n========== READ RAW DATA FROM HDFS ==========")

df = spark.read.csv(
    HDFS_INPUT_PATH,
    header=True,
    inferSchema=True
)

print_status("RAW DATASET", df)
record_feature_progress("Raw Dataset", df)

df_reduced = df
df_reduced.createOrReplaceTempView("prosper_loan_reduced")


# GROUP 0: Initial Domain Filtering
# Remove identifiers and post-origination fields that may cause leakage.
cols_group_0 = [
    "ListingKey",
    "ListingNumber",
    "LoanKey",
    "LoanNumber",
    "MemberKey",
    "GroupKey",
    "LP_CustomerPayments",
    "LP_CustomerPrincipalPayments",
    "LP_InterestandFees",
    "LP_ServiceFees",
    "LP_CollectionFees",
    "LP_GrossPrincipalLoss",
    "LP_NetPrincipalLoss",
    "LP_NonPrincipalRecoverypayments",
    "LoanCurrentDaysDelinquent",
    "LoanFirstDefaultedCycleNumber",
    "LoanMonthsSinceOrigination",
    "CurrentlyInGroup",
    "ClosedDate"
]

group_0_columns_before = len(df_reduced.columns)
df_reduced = safe_drop(df_reduced, cols_group_0)
df_reduced.createOrReplaceTempView("prosper_loan_reduced")
print_status("GROUP 0 - AFTER INITIAL DOMAIN FILTERING", df_reduced)
record_feature_progress("Initial Domain Filtering", df_reduced)
print_initial_domain_filtering_summary(
    group_0_columns_before,
    group_0_columns_before - len(df_reduced.columns),
    len(df_reduced.columns)
)


# GROUP 1: Internal Risk Metrics
# Check whether ProsperScore summarizes internal pricing and risk signals.
print("\n========== GROUP 1 - INTERNAL RISK METRICS ==========")

spark.sql("""
SELECT
    ProsperScore,
    COUNT(*) AS total_loans,
    ROUND(AVG(BorrowerAPR), 4) AS avg_apr,
    ROUND(
        AVG(
            CASE
                WHEN LoanStatus IN ('Chargedoff', 'Defaulted') THEN 1
                WHEN LoanStatus = 'Completed' THEN 0
                ELSE NULL
            END
        ) * 100, 2
    ) AS bad_loan_rate_percent
FROM prosper_loan_reduced
WHERE ProsperScore IS NOT NULL
GROUP BY ProsperScore
ORDER BY ProsperScore
""").show(truncate=False)

print("Conclusion: ProsperScore is retained because it captures both loan pricing and credit risk patterns.")

cols_group_1 = [
    "ProsperRating (Alpha)",
    "ProsperRating (numeric)",
    "CreditGrade"
]

df_reduced = safe_drop(df_reduced, cols_group_1)
df_reduced.createOrReplaceTempView("prosper_loan_reduced")
print_status("GROUP 1 - AFTER DROP", df_reduced)
record_feature_progress("After Group 1 - Internal Risk Metrics", df_reduced)


# GROUP 2: Income and Employment
# Check borrower repayment capacity through income and employment status.
print("\n========== GROUP 2 - INCOME AND EMPLOYMENT ==========")

spark.sql("""
SELECT
    IncomeRange,
    EmploymentStatus,
    COUNT(*) AS total_loans,
    ROUND(AVG(BorrowerAPR), 4) AS avg_apr,
    ROUND(
        AVG(
            CASE
                WHEN LoanStatus IN ('Chargedoff', 'Defaulted') THEN 1
                WHEN LoanStatus = 'Completed' THEN 0
                ELSE NULL
            END
        ) * 100, 2
    ) AS bad_loan_rate_percent
FROM prosper_loan_reduced
WHERE IncomeRange IS NOT NULL
  AND EmploymentStatus IS NOT NULL
GROUP BY IncomeRange, EmploymentStatus
HAVING COUNT(*) >= 100
ORDER BY IncomeRange, EmploymentStatus
""").show(100, truncate=False)

print("Conclusion: IncomeRange and EmploymentStatus are retained because they represent borrower repayment capacity.")

cols_group_2 = [
    "IncomeVerifiable",
    "EmploymentStatusDuration",
    "StatedMonthlyIncome"
]

df_reduced = safe_drop(df_reduced, cols_group_2)
df_reduced.createOrReplaceTempView("prosper_loan_reduced")
print_status("GROUP 2 - AFTER DROP", df_reduced)
record_feature_progress("After Group 2 - Income and Employment", df_reduced)


# GROUP 3: Credit History
# Compare credit score and DTI as complementary credit risk dimensions.
print("\n========== GROUP 3 - CREDIT HISTORY ==========")

spark.sql("""
SELECT
    CASE
        WHEN CreditScoreRangeLower < 600 THEN '<600'
        WHEN CreditScoreRangeLower < 660 THEN '600-659'
        WHEN CreditScoreRangeLower < 720 THEN '660-719'
        ELSE '720+'
    END AS credit_score_group,
    CASE
        WHEN DebtToIncomeRatio < 0.10 THEN 'Low DTI'
        WHEN DebtToIncomeRatio < 0.30 THEN 'Medium DTI'
        ELSE 'High DTI'
    END AS dti_group,
    COUNT(*) AS total_loans,
    ROUND(AVG(BorrowerAPR), 4) AS avg_apr,
    ROUND(
        AVG(
            CASE
                WHEN LoanStatus IN ('Chargedoff', 'Defaulted') THEN 1
                WHEN LoanStatus = 'Completed' THEN 0
                ELSE NULL
            END
        ) * 100, 2
    ) AS bad_loan_rate_percent
FROM prosper_loan_reduced
WHERE CreditScoreRangeLower IS NOT NULL
  AND DebtToIncomeRatio IS NOT NULL
GROUP BY
    CASE
        WHEN CreditScoreRangeLower < 600 THEN '<600'
        WHEN CreditScoreRangeLower < 660 THEN '600-659'
        WHEN CreditScoreRangeLower < 720 THEN '660-719'
        ELSE '720+'
    END,
    CASE
        WHEN DebtToIncomeRatio < 0.10 THEN 'Low DTI'
        WHEN DebtToIncomeRatio < 0.30 THEN 'Medium DTI'
        ELSE 'High DTI'
    END
HAVING COUNT(*) >= 100
ORDER BY credit_score_group, dti_group
""").show(100, truncate=False)

print("Conclusion: CreditScoreRangeLower and DebtToIncomeRatio are retained because they capture different but complementary credit risk dimensions.")

cols_group_3 = [
    "CreditScoreRangeUpper",
    "OpenCreditLines",
    "OpenRevolvingAccounts"
]

df_reduced = safe_drop(df_reduced, cols_group_3)
df_reduced.createOrReplaceTempView("prosper_loan_reduced")
print_status("GROUP 3 - AFTER DROP", df_reduced)
record_feature_progress("After Group 3 - Credit History", df_reduced)


# GROUP 4: Debt and Credit Utilization
# Check credit utilization as a direct indicator of revolving credit pressure.
print("\n========== GROUP 4 - DEBT AND CREDIT UTILIZATION ==========")

spark.sql("""
SELECT
    CASE
        WHEN BankcardUtilization < 0.30 THEN 'Low Utilization'
        WHEN BankcardUtilization < 0.70 THEN 'Medium Utilization'
        ELSE 'High Utilization'
    END AS utilization_group,
    COUNT(*) AS total_loans,
    ROUND(AVG(BorrowerAPR), 4) AS avg_apr,
    ROUND(
        AVG(
            CASE
                WHEN LoanStatus IN ('Chargedoff', 'Defaulted') THEN 1
                WHEN LoanStatus = 'Completed' THEN 0
                ELSE NULL
            END
        ) * 100, 2
    ) AS bad_loan_rate_percent
FROM prosper_loan_reduced
WHERE BankcardUtilization IS NOT NULL
GROUP BY
    CASE
        WHEN BankcardUtilization < 0.30 THEN 'Low Utilization'
        WHEN BankcardUtilization < 0.70 THEN 'Medium Utilization'
        ELSE 'High Utilization'
    END
ORDER BY avg_apr DESC
""").show(truncate=False)

print("Conclusion: BankcardUtilization is retained because it reflects revolving credit pressure.")

cols_group_4 = [
    "AvailableBankcardCredit",
    "RevolvingCreditBalance",
    "OpenRevolvingMonthlyPayment"
]

df_reduced = safe_drop(df_reduced, cols_group_4)
df_reduced.createOrReplaceTempView("prosper_loan_reduced")
print_status("GROUP 4 - AFTER DROP", df_reduced)
record_feature_progress("After Group 4 - Debt and Credit Utilization", df_reduced)


# GROUP 5: Delinquency and Public Records
# Keep core delinquency history and remove weaker public-record indicators.
print("\n========== GROUP 5 - DELINQUENCY AND PUBLIC RECORDS ==========")

spark.sql("""
WITH base AS (
    SELECT
        CASE
            WHEN CurrentDelinquencies = 0 THEN 'No Current Delinquency'
            WHEN CurrentDelinquencies <= 2 THEN '1-2 Current Delinquencies'
            ELSE '3+ Current Delinquencies'
        END AS current_delinquency_group,

        CASE
            WHEN DelinquenciesLast7Years = 0 THEN 'No 7Y Delinquency'
            WHEN DelinquenciesLast7Years <= 2 THEN '1-2 in 7Y'
            WHEN DelinquenciesLast7Years <= 5 THEN '3-5 in 7Y'
            ELSE '5+ in 7Y'
        END AS delinquency_history_group,

        BorrowerAPR,
        LoanStatus,
        AmountDelinquent,
        PublicRecordsLast10Years,
        PublicRecordsLast12Months
    FROM prosper_loan_reduced
    WHERE CurrentDelinquencies IS NOT NULL
      AND DelinquenciesLast7Years IS NOT NULL
      AND BorrowerAPR IS NOT NULL
),
agg AS (
    SELECT
        current_delinquency_group,
        delinquency_history_group,
        COUNT(*) AS total_loans,

        ROUND(AVG(BorrowerAPR), 4) AS avg_apr,

        ROUND(
            AVG(
                CASE
                    WHEN LoanStatus IN ('Chargedoff', 'Defaulted') THEN 1.0
                    WHEN LoanStatus = 'Completed' THEN 0.0
                    ELSE NULL
                END
            ) * 100, 2
        ) AS bad_loan_rate_percent,

        ROUND(AVG(COALESCE(AmountDelinquent, 0)), 2) AS avg_amount_delinquent,
        ROUND(AVG(COALESCE(PublicRecordsLast10Years, 0)), 2) AS avg_public_records_10y,
        ROUND(AVG(COALESCE(PublicRecordsLast12Months, 0)), 2) AS avg_public_records_12m
    FROM base
    GROUP BY current_delinquency_group, delinquency_history_group
    HAVING COUNT(*) >= 100
)
SELECT
    *,
    DENSE_RANK() OVER (ORDER BY avg_apr DESC) AS apr_risk_rank,
    DENSE_RANK() OVER (
        ORDER BY COALESCE(bad_loan_rate_percent, -1) DESC
    ) AS default_risk_rank
FROM agg
ORDER BY apr_risk_rank, default_risk_rank
""").show(50, truncate=False)

print(
    "Conclusion: CurrentDelinquencies and DelinquenciesLast7Years are retained because "
    "they jointly capture current and historical delinquency risk. AmountDelinquent and "
    "PublicRecordsLast12Months are removed as weaker or overlapping delinquency-related signals."
)

cols_group_5 = [
    "AmountDelinquent",
    "PublicRecordsLast12Months"
]

df_reduced = safe_drop(df_reduced, cols_group_5)
df_reduced.createOrReplaceTempView("prosper_loan_reduced")
print_status("GROUP 5 - AFTER DROP", df_reduced)
record_feature_progress("After Group 5 - Delinquency and Public Records", df_reduced)


# GROUP 6: Previous Prosper Borrowing History
# Retain prior Prosper borrowing experience and late-payment behavior.
print("\n========== GROUP 6 - PREVIOUS PROSPER BORROWING HISTORY ==========")

spark.sql("""
WITH base AS (
    SELECT
        CASE
            WHEN COALESCE(TotalProsperLoans, 0) = 0 THEN 'No Previous Loan'
            WHEN COALESCE(TotalProsperLoans, 0) = 1 THEN '1 Previous Loan'
            ELSE '2+ Previous Loans'
        END AS prosper_loan_history,

        CASE
            WHEN COALESCE(ProsperPaymentsOneMonthPlusLate, 0) = 0
                THEN 'No 1M+ Late Payment'
            ELSE 'Has 1M+ Late Payment'
        END AS prosper_late_payment_group,

        BorrowerAPR,
        LoanStatus,
        TotalProsperPaymentsBilled,
        OnTimeProsperPayments,
        ProsperPrincipalBorrowed,
        ProsperPrincipalOutstanding
    FROM prosper_loan_reduced
    WHERE BorrowerAPR IS NOT NULL
),
agg AS (
    SELECT
        prosper_loan_history,
        prosper_late_payment_group,
        COUNT(*) AS total_loans,

        ROUND(AVG(BorrowerAPR), 4) AS avg_apr,

        ROUND(
            AVG(
                CASE
                    WHEN LoanStatus IN ('Chargedoff', 'Defaulted') THEN 1.0
                    WHEN LoanStatus = 'Completed' THEN 0.0
                    ELSE NULL
                END
            ) * 100, 2
        ) AS bad_loan_rate_percent,

        ROUND(AVG(COALESCE(OnTimeProsperPayments, 0)), 2) AS avg_on_time_payments,
        ROUND(AVG(COALESCE(TotalProsperPaymentsBilled, 0)), 2) AS avg_payments_billed,
        ROUND(AVG(COALESCE(ProsperPrincipalBorrowed, 0)), 2) AS avg_principal_borrowed,
        ROUND(AVG(COALESCE(ProsperPrincipalOutstanding, 0)), 2) AS avg_principal_outstanding
    FROM base
    GROUP BY prosper_loan_history, prosper_late_payment_group
    HAVING COUNT(*) >= 100
)
SELECT
    *,
    DENSE_RANK() OVER (ORDER BY avg_apr DESC) AS apr_risk_rank,
    DENSE_RANK() OVER (
        ORDER BY COALESCE(bad_loan_rate_percent, -1) DESC
    ) AS default_risk_rank
FROM agg
ORDER BY apr_risk_rank, default_risk_rank
""").show(50, truncate=False)

print(
    "Conclusion: TotalProsperLoans is retained to represent prior platform experience, "
    "and ProsperPaymentsOneMonthPlusLate is retained as a direct late-payment risk signal. "
    "Payment count and principal amount variables are removed because they mainly describe "
    "transaction volume and may overlap with prior borrowing history."
)

cols_group_6 = [
    "TotalProsperPaymentsBilled",
    "OnTimeProsperPayments",
    "ProsperPrincipalBorrowed",
    "ProsperPrincipalOutstanding"
]

df_reduced = safe_drop(df_reduced, cols_group_6)
df_reduced.createOrReplaceTempView("prosper_loan_reduced")
print_status("GROUP 6 - AFTER DROP", df_reduced)
record_feature_progress("After Group 6 - Previous Prosper Borrowing History", df_reduced)


# GROUP 7: Loan Structure and Funding
# Keep core loan size, term, and investor participation features.
print("\n========== GROUP 7 - LOAN STRUCTURE AND FUNDING CHECK ==========")

spark.sql("""
WITH base AS (
    SELECT
        CASE
            WHEN LoanOriginalAmount < 5000 THEN 'Small Loan'
            WHEN LoanOriginalAmount < 15000 THEN 'Medium Loan'
            ELSE 'Large Loan'
        END AS loan_size_group,

        Term,

        CASE
            WHEN Investors < 50 THEN 'Low Investor Interest'
            WHEN Investors < 200 THEN 'Medium Investor Interest'
            ELSE 'High Investor Interest'
        END AS investor_group,

        BorrowerAPR,
        LoanStatus,
        MonthlyLoanPayment,
        PercentFunded,
        Recommendations,
        InvestmentFromFriendsCount,
        InvestmentFromFriendsAmount
    FROM prosper_loan_reduced
    WHERE LoanOriginalAmount IS NOT NULL
      AND Term IS NOT NULL
      AND Investors IS NOT NULL
      AND BorrowerAPR IS NOT NULL
),
agg AS (
    SELECT
        loan_size_group,
        Term,
        investor_group,
        COUNT(*) AS total_loans,

        ROUND(AVG(BorrowerAPR), 4) AS avg_apr,

        ROUND(
            AVG(
                CASE
                    WHEN LoanStatus IN ('Chargedoff', 'Defaulted') THEN 1.0
                    WHEN LoanStatus = 'Completed' THEN 0.0
                    ELSE NULL
                END
            ) * 100, 2
        ) AS bad_loan_rate_percent,

        ROUND(AVG(MonthlyLoanPayment), 2) AS avg_monthly_payment,
        ROUND(AVG(PercentFunded), 4) AS avg_percent_funded,
        ROUND(AVG(Recommendations), 2) AS avg_recommendations,
        ROUND(AVG(InvestmentFromFriendsCount), 2) AS avg_friend_invest_count,
        ROUND(AVG(InvestmentFromFriendsAmount), 2) AS avg_friend_invest_amount
    FROM base
    GROUP BY loan_size_group, Term, investor_group
    HAVING COUNT(*) >= 100
)
SELECT
    *,
    DENSE_RANK() OVER (ORDER BY avg_apr DESC) AS apr_rank,
    DENSE_RANK() OVER (
        ORDER BY COALESCE(bad_loan_rate_percent, -1) DESC
    ) AS default_risk_rank
FROM agg
ORDER BY apr_rank, default_risk_rank
""").show(80, truncate=False)

print(
    "Conclusion: LoanOriginalAmount, Term, and Investors are retained because they represent "
    "loan size, contract structure, and market funding interest. MonthlyLoanPayment, "
    "Recommendations, InvestmentFromFriendsCount, and InvestmentFromFriendsAmount are removed "
    "because they are derivative, sparse, or provide limited additional business value."
)

cols_group_7 = [
    "MonthlyLoanPayment",
    "Recommendations",
    "InvestmentFromFriendsCount",
    "InvestmentFromFriendsAmount"
]

df_reduced = safe_drop(df_reduced, cols_group_7)
df_reduced.createOrReplaceTempView("prosper_loan_reduced")
print_status("GROUP 7 - AFTER DROP", df_reduced)
record_feature_progress("After Group 7 - Loan Structure and Funding", df_reduced)


# GROUP 8: Estimated Pricing Components
# Remove pricing-output variables while retaining EstimatedLoss as the core expected risk signal.
print("\n========== GROUP 8 - ESTIMATED LOSS AND PRICING LEAKAGE CHECK ==========")

spark.sql("""
WITH corr_check AS (
    SELECT
        ROUND(corr(BorrowerAPR, BorrowerRate), 4) AS corr_apr_borrower_rate,
        ROUND(corr(BorrowerAPR, LenderYield), 4) AS corr_apr_lender_yield,
        ROUND(corr(BorrowerAPR, EstimatedEffectiveYield), 4) AS corr_apr_effective_yield,
        ROUND(corr(BorrowerAPR, EstimatedReturn), 4) AS corr_apr_estimated_return,
        ROUND(corr(EstimatedLoss, EstimatedReturn), 4) AS corr_loss_return
    FROM prosper_loan_reduced
),
base AS (
    SELECT
        CASE
            WHEN EstimatedLoss < 0.05 THEN 'Low Expected Loss'
            WHEN EstimatedLoss < 0.10 THEN 'Medium Expected Loss'
            ELSE 'High Expected Loss'
        END AS expected_loss_group,

        BorrowerAPR,
        LoanStatus,
        EstimatedLoss,
        BorrowerRate,
        LenderYield,
        EstimatedEffectiveYield,
        EstimatedReturn
    FROM prosper_loan_reduced
    WHERE EstimatedLoss IS NOT NULL
      AND BorrowerAPR IS NOT NULL
),
agg AS (
    SELECT
        expected_loss_group,
        COUNT(*) AS total_loans,

        ROUND(AVG(BorrowerAPR), 4) AS avg_apr,
        ROUND(AVG(EstimatedLoss), 4) AS avg_estimated_loss,
        ROUND(AVG(BorrowerRate), 4) AS avg_borrower_rate,
        ROUND(AVG(LenderYield), 4) AS avg_lender_yield,
        ROUND(AVG(EstimatedEffectiveYield), 4) AS avg_effective_yield,
        ROUND(AVG(EstimatedReturn), 4) AS avg_estimated_return,

        ROUND(
            AVG(
                CASE
                    WHEN LoanStatus IN ('Chargedoff', 'Defaulted') THEN 1.0
                    WHEN LoanStatus = 'Completed' THEN 0.0
                    ELSE NULL
                END
            ) * 100, 2
        ) AS bad_loan_rate_percent
    FROM base
    GROUP BY expected_loss_group
)
SELECT
    agg.*,
    corr_check.corr_apr_borrower_rate,
    corr_check.corr_apr_lender_yield,
    corr_check.corr_apr_effective_yield,
    corr_check.corr_apr_estimated_return,
    corr_check.corr_loss_return,
    DENSE_RANK() OVER (ORDER BY avg_apr DESC) AS apr_rank,
    DENSE_RANK() OVER (
        ORDER BY COALESCE(bad_loan_rate_percent, -1) DESC
    ) AS default_risk_rank
FROM agg
CROSS JOIN corr_check
ORDER BY apr_rank, default_risk_rank
""").show(truncate=False)

print(
    "Conclusion: EstimatedLoss is retained because it represents expected credit risk. "
    "BorrowerRate, LenderYield, EstimatedEffectiveYield, and EstimatedReturn are removed "
    "because they are pricing-output variables that are highly related to BorrowerAPR and may "
    "introduce target leakage or redundant pricing information in the APR prediction task."
)

cols_group_8 = [
    "BorrowerRate",
    "LenderYield",
    "EstimatedEffectiveYield",
    "EstimatedReturn"
]

df_reduced = safe_drop(df_reduced, cols_group_8)
df_reduced.createOrReplaceTempView("prosper_loan_reduced")
print_status("GROUP 8 - AFTER PRICING COMPONENTS REDUCTION", df_reduced)
record_feature_progress("After Group 8 - Estimated Pricing Components", df_reduced)


# GROUP 9: Credit Capacity and Trade Structure
# Keep compact credit capacity signals and remove overlapping trade history fields.
print("\n========== GROUP 9 - DELINQUENCY AND CREDIT HISTORY RISK CHECK ==========")

spark.sql("""
SELECT
    CASE
        WHEN CurrentDelinquencies = 0 THEN 'No Current Delinquency'
        WHEN CurrentDelinquencies <= 2 THEN '1-2 Current Delinquencies'
        ELSE '3+ Current Delinquencies'
    END AS delinquency_group,

    CASE
        WHEN TotalTrades < 10 THEN 'Thin Credit History'
        WHEN TotalTrades < 30 THEN 'Medium Credit History'
        ELSE 'Strong Credit History'
    END AS credit_history_group,

    COUNT(*) AS total_loans,
    ROUND(AVG(BorrowerAPR), 4) AS avg_apr,

    ROUND(
        SUM(CASE
            WHEN LoanStatus IN ('Chargedoff', 'Defaulted') THEN 1
            ELSE 0
        END) / COUNT(*),
        4
    ) AS bad_loan_rate

FROM prosper_loan_reduced
WHERE CurrentDelinquencies IS NOT NULL
  AND TotalTrades IS NOT NULL
  AND BorrowerAPR IS NOT NULL
  AND LoanStatus IS NOT NULL
GROUP BY
    CASE
        WHEN CurrentDelinquencies = 0 THEN 'No Current Delinquency'
        WHEN CurrentDelinquencies <= 2 THEN '1-2 Current Delinquencies'
        ELSE '3+ Current Delinquencies'
    END,
    CASE
        WHEN TotalTrades < 10 THEN 'Thin Credit History'
        WHEN TotalTrades < 30 THEN 'Medium Credit History'
        ELSE 'Strong Credit History'
    END
ORDER BY bad_loan_rate DESC, avg_apr DESC
""").show(truncate=False)


spark.sql("""
SELECT
    ROUND(corr(CurrentCreditLines, TotalCreditLinespast7years), 4) AS corr_credit_history,
    ROUND(corr(CurrentCreditLines, TotalTrades), 4) AS corr_credit_trade,
    ROUND(corr(TotalTrades, `TradesNeverDelinquent (percentage)`), 4) AS corr_trade_quality
FROM prosper_loan_reduced
""").show(truncate=False)


cols_group_9 = [
    "TradesNeverDelinquent (percentage)",
    "PublicRecordsLast10Years",
    "TotalCreditLinespast7years"
]

df_reduced = safe_drop(df_reduced, cols_group_9)
df_reduced.createOrReplaceTempView("prosper_loan_reduced")
print_status("GROUP 9 - AFTER CREDIT CAPACITY REDUCTION", df_reduced)
record_feature_progress("After Group 9 - Credit Capacity and Trade Structure", df_reduced)

# GROUP 10: Borrower Profile and Demographic Features
# Remove high-cardinality borrower descriptors and keep home ownership as the compact profile signal.
print("\n========== GROUP 10 - BORROWER PROFILE AND DEMOGRAPHIC FEATURES ==========")

spark.sql("""
SELECT
    CASE
        WHEN IsBorrowerHomeowner = true THEN 'Homeowner'
        ELSE 'Non-Homeowner'
    END AS homeowner_group,
    COUNT(*) AS total_loans,
    ROUND(AVG(BorrowerAPR), 4) AS avg_apr
FROM prosper_loan_reduced
WHERE IsBorrowerHomeowner IS NOT NULL
GROUP BY
    CASE
        WHEN IsBorrowerHomeowner = true THEN 'Homeowner'
        ELSE 'Non-Homeowner'
    END
ORDER BY avg_apr DESC
""").show(truncate=False)

spark.sql("""
SELECT
    EmploymentStatus,
    COUNT(*) AS total_loans,
    ROUND(AVG(BorrowerAPR), 4) AS avg_apr
FROM prosper_loan_reduced
WHERE EmploymentStatus IS NOT NULL
GROUP BY EmploymentStatus
HAVING COUNT(*) > 500
ORDER BY avg_apr DESC
""").show(truncate=False)

spark.sql("""
SELECT
    CASE
        WHEN IsBorrowerHomeowner = true THEN 'Homeowner'
        ELSE 'Non-Homeowner'
    END AS homeowner_group,
    EmploymentStatus,
    COUNT(*) AS total_loans,
    ROUND(AVG(BorrowerAPR), 4) AS avg_apr
FROM prosper_loan_reduced
WHERE IsBorrowerHomeowner IS NOT NULL
  AND EmploymentStatus IS NOT NULL
GROUP BY
    CASE
        WHEN IsBorrowerHomeowner = true THEN 'Homeowner'
        ELSE 'Non-Homeowner'
    END,
    EmploymentStatus
HAVING COUNT(*) > 200
ORDER BY total_loans DESC
""").show(truncate=False)

spark.sql("""
SELECT
    BorrowerState,
    COUNT(*) AS total_loans,
    ROUND(AVG(BorrowerAPR), 4) AS avg_apr
FROM prosper_loan_reduced
WHERE BorrowerState IS NOT NULL
GROUP BY BorrowerState
HAVING COUNT(*) > 1000
ORDER BY total_loans DESC
LIMIT 15
""").show(truncate=False)

spark.sql("""
SELECT
    Occupation,
    COUNT(*) AS total_loans,
    ROUND(AVG(BorrowerAPR), 4) AS avg_apr
FROM prosper_loan_reduced
WHERE Occupation IS NOT NULL
GROUP BY Occupation
HAVING COUNT(*) > 500
ORDER BY total_loans DESC
LIMIT 15
""").show(truncate=False)

print("Conclusion: IsBorrowerHomeowner is retained, while BorrowerState, Occupation, and EmploymentStatus are removed to reduce weak or redundant profile information.")

cols_group_10 = [
    "BorrowerState",
    "Occupation",
    "EmploymentStatus"
]

df_reduced = safe_drop(df_reduced, cols_group_10)
df_reduced.createOrReplaceTempView("prosper_loan_reduced")
print_status("GROUP 10 - AFTER DROP", df_reduced)
record_feature_progress("After Group 10 - Borrower Profile", df_reduced)


# GROUP 11: Credit Inquiry and Credit Activity Features
# Keep recent inquiry and trade-depth signals, remove overlapping credit activity variables.
print("\n========== GROUP 11 - CREDIT INQUIRY AND CREDIT ACTIVITY FEATURES ==========")

spark.sql("""
WITH corr_check AS (
    SELECT
        ROUND(corr(InquiriesLast6Months, TotalInquiries), 4) AS corr_recent_total_inquiries,
        ROUND(corr(CurrentCreditLines, TotalTrades), 4) AS corr_current_lines_total_trades,
        ROUND(corr(TradesOpenedLast6Months, InquiriesLast6Months), 4) AS corr_recent_trades_inquiries
    FROM prosper_loan_reduced
),
base AS (
    SELECT
        CASE
            WHEN InquiriesLast6Months = 0 THEN 'No Recent Inquiry'
            WHEN InquiriesLast6Months <= 2 THEN '1-2 Recent Inquiries'
            ELSE '3+ Recent Inquiries'
        END AS recent_inquiry_group,

        CASE
            WHEN TotalTrades < 10 THEN 'Thin Credit History'
            WHEN TotalTrades < 30 THEN 'Medium Credit History'
            ELSE 'Long Credit History'
        END AS trade_history_group,

        BorrowerAPR,
        LoanStatus,
        TotalInquiries,
        TradesOpenedLast6Months,
        CurrentCreditLines,
        TotalTrades
    FROM prosper_loan_reduced
    WHERE InquiriesLast6Months IS NOT NULL
      AND TotalTrades IS NOT NULL
      AND BorrowerAPR IS NOT NULL
),
agg AS (
    SELECT
        recent_inquiry_group,
        trade_history_group,
        COUNT(*) AS total_loans,

        ROUND(AVG(BorrowerAPR), 4) AS avg_apr,

        ROUND(
            AVG(
                CASE
                    WHEN LoanStatus IN ('Chargedoff', 'Defaulted') THEN 1.0
                    WHEN LoanStatus = 'Completed' THEN 0.0
                    ELSE NULL
                END
            ) * 100, 2
        ) AS bad_loan_rate_percent,

        ROUND(AVG(TotalInquiries), 2) AS avg_total_inquiries,
        ROUND(AVG(TradesOpenedLast6Months), 2) AS avg_recent_trades_opened,
        ROUND(AVG(CurrentCreditLines), 2) AS avg_current_credit_lines
    FROM base
    GROUP BY recent_inquiry_group, trade_history_group
    HAVING COUNT(*) >= 100
)
SELECT
    agg.*,
    corr_check.corr_recent_total_inquiries,
    corr_check.corr_current_lines_total_trades,
    corr_check.corr_recent_trades_inquiries,
    DENSE_RANK() OVER (ORDER BY avg_apr DESC) AS apr_rank,
    DENSE_RANK() OVER (
        ORDER BY COALESCE(bad_loan_rate_percent, -1) DESC
    ) AS default_risk_rank
FROM agg
CROSS JOIN corr_check
ORDER BY apr_rank, default_risk_rank
""").show(80, truncate=False)

print(
    "Conclusion: InquiriesLast6Months and TotalTrades are retained because they represent "
    "recent credit-seeking behavior and the depth of credit history. TotalInquiries, "
    "CurrentCreditLines, and TradesOpenedLast6Months are removed because they overlap with "
    "the retained inquiry and trade-history signals."
)

cols_group_11 = [
    "TotalInquiries",
    "CurrentCreditLines",
    "TradesOpenedLast6Months"
]

df_reduced = safe_drop(df_reduced, cols_group_11)
df_reduced.createOrReplaceTempView("prosper_loan_reduced")
print_status("GROUP 11 - AFTER DROP", df_reduced)
record_feature_progress("After Group 11 - Credit Inquiry and Activity", df_reduced)


# GROUP 12: Temporal and Listing Information Redundancy Analysis
# Retain Investors as the stronger funding signal and remove weak descriptive listing/time variables.
print("\n========== GROUP 12 - TEMPORAL AND LISTING INFORMATION REDUNDANCY ANALYSIS ==========")

spark.sql("""
WITH base AS (
    SELECT
        YEAR(LoanOriginationDate) AS orig_year,
        QUARTER(LoanOriginationDate) AS orig_quarter,

        `ListingCategory (numeric)` AS listing_category,

        CASE
            WHEN PercentFunded >= 1.0 THEN 'Fully Funded'
            WHEN PercentFunded >= 0.8 THEN 'Mostly Funded'
            ELSE 'Low or Medium Funded'
        END AS funding_group,

        BorrowerAPR,
        LoanStatus,
        ScorexChangeAtTimeOfListing,
        PercentFunded,
        Investors
    FROM prosper_loan_reduced
    WHERE LoanOriginationDate IS NOT NULL
      AND `ListingCategory (numeric)` IS NOT NULL
      AND PercentFunded IS NOT NULL
      AND BorrowerAPR IS NOT NULL
),
agg AS (
    SELECT
        orig_year,
        orig_quarter,
        listing_category,
        funding_group,
        COUNT(*) AS total_loans,

        ROUND(AVG(BorrowerAPR), 4) AS avg_apr,

        ROUND(
            AVG(
                CASE
                    WHEN LoanStatus IN ('Chargedoff', 'Defaulted') THEN 1.0
                    WHEN LoanStatus = 'Completed' THEN 0.0
                    ELSE NULL
                END
            ) * 100, 2
        ) AS bad_loan_rate_percent,

        ROUND(AVG(PercentFunded), 4) AS avg_percent_funded,
        ROUND(AVG(Investors), 2) AS avg_investors,
        ROUND(AVG(ScorexChangeAtTimeOfListing), 2) AS avg_score_change,

        ROUND(
            AVG(
                CASE
                    WHEN ScorexChangeAtTimeOfListing < 0 THEN 1.0
                    ELSE 0.0
                END
            ) * 100, 2
        ) AS score_decrease_percent
    FROM base
    GROUP BY orig_year, orig_quarter, listing_category, funding_group
    HAVING COUNT(*) >= 100
),
time_check AS (
    SELECT
        *,
        LAG(avg_apr) OVER (
            PARTITION BY listing_category, funding_group
            ORDER BY orig_year, orig_quarter
        ) AS prev_quarter_avg_apr
    FROM agg
)
SELECT
    *,
    ROUND(avg_apr - prev_quarter_avg_apr, 4) AS apr_change_vs_prev_quarter,
    DENSE_RANK() OVER (
        PARTITION BY orig_year, orig_quarter
        ORDER BY avg_apr DESC
    ) AS quarterly_apr_rank
FROM time_check
ORDER BY orig_year, orig_quarter, quarterly_apr_rank
""").show(100, truncate=False)

spark.sql("""
SELECT
    ROUND(corr(PercentFunded, Investors), 4) AS corr_percent_funded_investors
FROM prosper_loan_reduced
WHERE PercentFunded IS NOT NULL
  AND Investors IS NOT NULL
""").show(truncate=False)

print(
    "Conclusion: Investors is retained as the stronger funding-market signal. "
    "ListingCategory (numeric), PercentFunded, LoanOriginationQuarter, and "
    "ScorexChangeAtTimeOfListing are removed because they are weak, unstable, or mostly "
    "descriptive listing and temporal variables after stronger credit-risk features are retained."
)

cols_group_12 = [
    "ListingCategory (numeric)",
    "PercentFunded",
    "LoanOriginationQuarter",
    "ScorexChangeAtTimeOfListing"
]

df_reduced = safe_drop(df_reduced, cols_group_12)
df_reduced.createOrReplaceTempView("prosper_loan_reduced")
print_status("GROUP 12 - AFTER DROP", df_reduced)
record_feature_progress("Final Reduced Dataset", df_reduced)


print_feature_reduction_progress()
print("\nConclusion:")
print(
    "The feature analysis workflow reduced the dataset from the original "
    f"{feature_progress[0][1]} attributes to the final reduced feature set."
)
print(
    "This step helps remove identifiers, redundant variables, high-leakage attributes, "
    "and weak business-value features before EDA, preprocessing, and machine learning."
)


print("\n========== REMAINING COLUMNS ==========")
for index, column_name in enumerate(df_reduced.columns, start=1):
    print(f"{index:02d}. {column_name}")

print("\n========== SAVE REDUCED DATASET TO HDFS ==========")
df_reduced.write.mode("overwrite").parquet(HDFS_OUTPUT_PATH)

print(f"Saved reduced dataset to: {HDFS_OUTPUT_PATH}")
print(f"Remaining rows: {df_reduced.count()}")
print(f"Remaining columns: {len(df_reduced.columns)}")
print("Domain feature reduction completed successfully.")

spark.stop()



========== READ RAW DATA FROM HDFS ==========

========== RAW DATASET ==========
Rows: 113937
Columns: 81
Dropped columns (19):
- ListingKey
- ListingNumber
- LoanKey
- LoanNumber
- MemberKey
- GroupKey
- LP_CustomerPayments
- LP_CustomerPrincipalPayments
- LP_InterestandFees
- LP_ServiceFees
- LP_CollectionFees
- LP_GrossPrincipalLoss
- LP_NetPrincipalLoss
- LP_NonPrincipalRecoverypayments
- LoanCurrentDaysDelinquent
- LoanFirstDefaultedCycleNumber
- LoanMonthsSinceOrigination
- CurrentlyInGroup
- ClosedDate

========== GROUP 0 - AFTER INITIAL DOMAIN FILTERING ==========
Rows: 113937
Columns: 62

========== INITIAL DOMAIN FILTERING SUMMARY ==========
Columns before filtering: 81
Columns removed: 19
Columns remaining: 62

Removed column categories:
- Identifier columns
- Duplicate loan identifiers
- Internal platform tracking fields
- Historical payment/recovery accounting fields
- Post-loan outcome leakage variables

Reason:
These columns are not suitable as predictive features becau

In [9]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


HDFS_REDUCED_DATASET_PATH = (
    "hdfs://localhost:9000/bigdata/prosper_loan/processed/prosper_loan_reduced"
)
HDFS_EDA_REDUCED_DATASET_PATH = (
    "hdfs://localhost:9000/bigdata/prosper_loan/processed/prosper_loan_eda_reduced"
)

OUTPUT_DIR = os.path.join("outputs", "charts")

income_order = [
    "Not displayed",
    "Not employed",
    "$0",
    "$1-24,999",
    "$25,000-49,999",
    "$50,000-74,999",
    "$75,000-99,999",
    "$100,000+",
]

numeric_type_names = {
    "byte",
    "short",
    "int",
    "bigint",
    "long",
    "float",
    "double",
}

manual_remove_reason_by_feature = {
    "BorrowerRate": (
        "Manual removal: overlaps strongly with BorrowerAPR and is excluded as a "
        "risk-pricing proxy."
    ),
    "LP_CustomerPayments": (
        "Manual removal: post-loan payment accounting field that may leak outcome "
        "information."
    ),
    "LP_CustomerPrincipalPayments": (
        "Manual removal: post-loan principal payment field that may leak outcome "
        "information."
    ),
    "LP_InterestandFees": (
        "Manual removal: post-loan interest and fee accounting field."
    ),
    "LP_ServiceFees": (
        "Manual removal: post-loan service fee accounting field."
    ),
    "LP_CollectionFees": (
        "Manual removal: post-loan collection fee field related to loan performance."
    ),
    "LP_GrossPrincipalLoss": (
        "Manual removal: loss field that directly reflects post-loan outcome."
    ),
    "LP_NetPrincipalLoss": (
        "Manual removal: loss field that directly reflects post-loan outcome."
    ),
    "LP_NonPrincipalRecoverypayments": (
        "Manual removal: recovery payment field observed after loan performance."
    ),
    "Investors": (
        "Manual removal: platform funding participation signal with limited borrower "
        "credit-risk business value for the EDA-reduced feature set."
    ),
    "Recommendations": (
        "Manual removal: platform/social signal with limited direct credit-risk "
        "business value."
    ),
}


def ensure_output_dirs():
    os.makedirs(OUTPUT_DIR, exist_ok=True)


def print_table_from_pandas(data):
    if data.empty:
        print("No rows to display.")
        return

    print(data.to_string(index=False))


def print_feature_list(title, features):
    print(f"{title} ({len(features)}):")
    if not features:
        print("- None")
        return

    for feature in features:
        print(f"- {feature}")


def calculate_missing_summary(df, total_rows, excluded_columns=None):
    excluded_columns = set(excluded_columns or [])
    columns = [column for column in df.columns if column not in excluded_columns]

    if not columns:
        return pd.DataFrame(columns=["column", "missing_count", "missing_percent"])

    null_expressions = [
        F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(column)
        for column in columns
    ]
    missing_counts = df.agg(*null_expressions).collect()[0].asDict()

    missing_summary = []
    for column in columns:
        missing_count = missing_counts[column]
        missing_percent = (
            (missing_count / total_rows) * 100
            if total_rows > 0
            else 0
        )
        missing_summary.append(
            {
                "column": column,
                "missing_count": missing_count,
                "missing_percent": round(missing_percent, 2),
            }
        )

    return (
        pd.DataFrame(missing_summary)
        .sort_values("missing_percent", ascending=False)
        .reset_index(drop=True)
    )


def get_numeric_features(df, excluded_columns=None):
    excluded_columns = set(excluded_columns or [])
    numeric_features = []

    for field in df.schema.fields:
        data_type = field.dataType.simpleString()
        is_numeric = data_type in numeric_type_names or data_type.startswith("decimal")
        if is_numeric and field.name not in excluded_columns:
            numeric_features.append(field.name)

    return numeric_features


def calculate_correlation_matrix(df, numeric_features):
    matrix = pd.DataFrame(
        np.eye(len(numeric_features)),
        index=numeric_features,
        columns=numeric_features,
    )

    for index_1, feature_1 in enumerate(numeric_features):
        for index_2 in range(index_1 + 1, len(numeric_features)):
            feature_2 = numeric_features[index_2]
            correlation = (
                df.select(F.corr(F.col(feature_1), F.col(feature_2)).alias("corr"))
                .collect()[0]["corr"]
            )

            if correlation is None:
                correlation = np.nan

            matrix.loc[feature_1, feature_2] = correlation
            matrix.loc[feature_2, feature_1] = correlation

    return matrix


def build_high_correlation_pairs(corr_matrix, threshold):
    corr_pairs = (
        corr_matrix
        .where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        .stack()
        .reset_index()
    )
    corr_pairs.columns = ["Feature_1", "Feature_2", "Correlation"]
    corr_pairs["AbsCorrelation"] = corr_pairs["Correlation"].abs()

    return (
        corr_pairs[corr_pairs["AbsCorrelation"] >= threshold]
        .sort_values("AbsCorrelation", ascending=False)
        .reset_index(drop=True)
    )


def add_removal_record(records, feature, stage, reason):
    if feature is None:
        return

    records.append(
        {
            "RemovedFeature": feature,
            "RemovalStage": stage,
            "Reason": reason,
        }
    )


def drop_features_if_present(df, features):
    existing_features = [feature for feature in features if feature in df.columns]
    if not existing_features:
        return df

    return df.drop(*existing_features)


def main():
    spark = (
        SparkSession.builder.appName("ProsperLoanEDAFeatureReduction")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")

    ensure_output_dirs()
    removal_records = []

    print("========== READ REDUCED DATASET ==========")
    df = spark.read.parquet(HDFS_REDUCED_DATASET_PATH)
    print(f"Reduced dataset loaded from: {HDFS_REDUCED_DATASET_PATH}")

    print("========== DATASET SUMMARY ==========")
    total_rows = df.count()
    original_columns = df.columns
    original_column_count = len(original_columns)

    print(f"Total rows: {total_rows}")
    print(f"Total columns: {original_column_count}")
    print("Columns:")
    for column in original_columns:
        print(f"- {column}")

    if "BorrowerRate" in original_columns:
        print(
            "Note: BorrowerRate exists in the current dataset but is ignored in EDA "
            "correlation analysis because it overlaps strongly with BorrowerAPR."
        )

    print("========== 4.1 DISTRIBUTION ANALYSIS ==========")

    print("========== LOAN OUTCOME DISTRIBUTION ==========")
    outcome_col = (
        F.when(F.col("LoanStatus") == "Completed", "Good Loan")
        .when(F.col("LoanStatus").isin("Chargedoff", "Defaulted"), "Bad Loan")
        .otherwise("Other / Not Finalized")
    )
    loan_outcome_pd = (
        df.withColumn("outcome_group", outcome_col)
        .groupBy("outcome_group")
        .count()
        .orderBy(F.col("count").desc())
        .toPandas()
    )
    if total_rows > 0:
        loan_outcome_pd["percentage"] = (
            loan_outcome_pd["count"] / total_rows * 100
        ).round(2)
    else:
        loan_outcome_pd["percentage"] = 0
    print_table_from_pandas(loan_outcome_pd[["outcome_group", "count", "percentage"]])

    plt.figure(figsize=(8, 8))
    plt.pie(
        loan_outcome_pd["count"],
        labels=loan_outcome_pd["outcome_group"],
        autopct="%1.1f%%",
        startangle=90,
        wedgeprops={"width": 0.45},
    )
    plt.title("Loan Outcome Distribution")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "loan_outcome_distribution.png"), dpi=300)
    plt.close()
    print(
        "Conclusion: The dataset contains a large proportion of active or non-finalized "
        "loans, so only final outcomes should be used later when constructing the "
        "classification target."
    )

    print("========== LOAN STATUS DISTRIBUTION ==========")
    loan_status_pd = (
        df.groupBy("LoanStatus")
        .count()
        .orderBy(F.col("count").desc())
        .toPandas()
    )
    print_table_from_pandas(loan_status_pd)

    plt.figure(figsize=(12, 6))
    sns.barplot(data=loan_status_pd, y="LoanStatus", x="count", color="#4C78A8")
    plt.title("LoanStatus Distribution")
    plt.xlabel("Count")
    plt.ylabel("LoanStatus")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "loan_status_distribution.png"), dpi=300)
    plt.close()
    print(
        "Conclusion: Current and Completed loans dominate the raw status distribution, "
        "while Defaulted and Chargedoff loans form a smaller but important risk group."
    )

    print("========== INCOME RANGE DISTRIBUTION ==========")
    income_pd = df.groupBy("IncomeRange").count().toPandas()
    income_pd["IncomeRange"] = pd.Categorical(
        income_pd["IncomeRange"],
        categories=income_order,
        ordered=True,
    )
    income_pd = income_pd.sort_values("IncomeRange").reset_index(drop=True)
    print_table_from_pandas(income_pd)

    plt.figure(figsize=(12, 6))
    sns.barplot(data=income_pd, x="IncomeRange", y="count", color="#59A14F")
    plt.title("IncomeRange Distribution")
    plt.xlabel("IncomeRange")
    plt.ylabel("Count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "income_range_distribution.png"), dpi=300)
    plt.close()
    print(
        "Conclusion: Borrowers are concentrated mainly in middle-income groups, while "
        "unemployed or zero-income groups are much smaller but may represent higher "
        "repayment risk."
    )

    print("========== PROSPER SCORE DISTRIBUTION ==========")
    prosper_score_pd = (
        df.groupBy("ProsperScore")
        .count()
        .orderBy(F.col("ProsperScore").asc_nulls_last())
        .toPandas()
    )
    print_table_from_pandas(prosper_score_pd)

    prosper_score_plot_pd = prosper_score_pd.copy()
    prosper_score_plot_pd["ProsperScoreLabel"] = (
        prosper_score_plot_pd["ProsperScore"].astype("string").fillna("Missing")
    )

    plt.figure(figsize=(12, 6))
    sns.barplot(
        data=prosper_score_plot_pd,
        x="ProsperScoreLabel",
        y="count",
        color="#F28E2B",
    )
    plt.plot(
        range(len(prosper_score_plot_pd)),
        prosper_score_plot_pd["count"],
        color="#1F2937",
        marker="o",
        linewidth=2,
    )
    plt.title("ProsperScore Distribution")
    plt.xlabel("ProsperScore")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "prosper_score_distribution.png"), dpi=300)
    plt.close()
    print(
        "Conclusion: ProsperScore is concentrated in the middle range, suggesting that "
        "most borrowers fall into moderate-risk groups."
    )

    print("========== MISSING VALUE DISTRIBUTION ==========")
    missing_summary_pd = calculate_missing_summary(
        df,
        total_rows,
        excluded_columns=["BorrowerRate"],
    )
    top_missing_pd = missing_summary_pd.head(10)
    print_table_from_pandas(top_missing_pd)

    plot_missing_pd = top_missing_pd.sort_values("missing_percent", ascending=True)
    plt.figure(figsize=(12, 6))
    sns.barplot(
        data=plot_missing_pd,
        x="missing_percent",
        y="column",
        color="#E15759",
    )
    plt.title("Top Missing Value Percentages by Feature")
    plt.xlabel("Missing Percent")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "missing_value_distribution.png"), dpi=300)
    plt.close()
    print(
        "Conclusion: Features with missing value percentages above 80% are strong "
        "candidates for removal during EDA-based feature reduction."
    )

    print("========== FEATURES REMOVED BY DISTRIBUTION/MISSING VALUE ANALYSIS ==========")
    distribution_candidate_features = [
        "TotalProsperLoans",
        "ProsperPaymentsLessThanOneMonthLate",
        "ProsperPaymentsOneMonthPlusLate",
    ]
    missing_percent_by_column = dict(
        zip(missing_summary_pd["column"], missing_summary_pd["missing_percent"])
    )
    distribution_removed_features = []

    for feature in distribution_candidate_features:
        missing_percent = missing_percent_by_column.get(feature)
        if missing_percent is not None and missing_percent >= 80:
            distribution_removed_features.append(feature)
            add_removal_record(
                removal_records,
                feature,
                "Distribution/Missing Value Analysis",
                "Missing value percentage above 80%.",
            )

    distribution_removal_pd = pd.DataFrame(
        [
            {
                "Feature": feature,
                "MissingPercent": missing_percent_by_column[feature],
                "Reason": "Missing value percentage above 80%.",
            }
            for feature in distribution_removed_features
        ],
        columns=["Feature", "MissingPercent", "Reason"],
    )
    print_table_from_pandas(distribution_removal_pd)
    print(
        "These features are removed because more than 80% of their values are missing, "
        "making them less reliable for downstream modeling."
    )

    df_after_distribution = drop_features_if_present(df, distribution_removed_features)

    print("========== 4.2 CORRELATION ANALYSIS ==========")
    print(
        "BorrowerAPR is used as a risk-pricing proxy for correlation analysis. "
        "Classification labels will be constructed later from LoanStatus."
    )

    print("========== MANUAL LEAKAGE OR PROXY REMOVAL BEFORE CORRELATION ==========")
    manual_remove_features = [
        feature
        for feature in manual_remove_reason_by_feature
        if feature in df_after_distribution.columns
    ]
    manual_removal_pd = pd.DataFrame(
        [
            {
                "Feature": feature,
                "Reason": manual_remove_reason_by_feature[feature],
            }
            for feature in manual_remove_features
        ],
        columns=["Feature", "Reason"],
    )
    print_table_from_pandas(manual_removal_pd)
    for feature in manual_remove_features:
        add_removal_record(
            removal_records,
            feature,
            "Manual Leakage or Proxy Removal",
            manual_remove_reason_by_feature[feature],
        )

    df_for_correlation = drop_features_if_present(
        df_after_distribution,
        manual_remove_features,
    )

    target_col = "BorrowerAPR"
    excluded_from_correlation = set(distribution_removed_features + manual_remove_features)
    excluded_from_correlation.add("BorrowerRate")
    numeric_features = get_numeric_features(
        df_for_correlation,
        excluded_columns=excluded_from_correlation,
    )

    print("Numerical features used for correlation analysis:")
    print_feature_list("Features", numeric_features)

    if target_col not in numeric_features:
        print(
            "BorrowerAPR is not available after EDA filtering, so Chapter 4.2 "
            "correlation analysis is skipped."
        )
        corr_matrix = pd.DataFrame()
        corr_with_apr = pd.DataFrame(
            columns=["Feature", "CorrelationWithBorrowerAPR", "AbsCorrelation"]
        )
        high_corr_pairs = pd.DataFrame(
            columns=["Feature_1", "Feature_2", "Correlation", "AbsCorrelation"]
        )
        redundant_remove_features = []
    else:
        corr_matrix = calculate_correlation_matrix(df_for_correlation, numeric_features)

        print("========== 4.2.1 Correlation with BorrowerAPR ==========")
        corr_with_apr = (
            corr_matrix[target_col]
            .drop(target_col)
            .dropna()
            .sort_values(key=lambda values: values.abs(), ascending=False)
            .reset_index()
        )
        corr_with_apr.columns = ["Feature", "CorrelationWithBorrowerAPR"]
        corr_with_apr["AbsCorrelation"] = (
            corr_with_apr["CorrelationWithBorrowerAPR"].abs()
        )

        print_table_from_pandas(corr_with_apr)
        corr_with_apr.to_csv(
            os.path.join(OUTPUT_DIR, "correlation_with_borrower_apr.csv"),
            index=False,
        )

        plot_data = corr_with_apr.sort_values("CorrelationWithBorrowerAPR")
        plt.figure(figsize=(10, 7))
        plt.barh(
            plot_data["Feature"],
            plot_data["CorrelationWithBorrowerAPR"],
            color="#4C78A8",
        )
        plt.axvline(0, linewidth=1, color="#1F2937")
        plt.title("Correlation with BorrowerAPR")
        plt.xlabel("Pearson Correlation")
        plt.ylabel("Feature")
        plt.tight_layout()
        plt.savefig(
            os.path.join(OUTPUT_DIR, "correlation_with_borrower_apr.png"),
            dpi=300,
            bbox_inches="tight",
        )
        plt.close()
        print("Saved chart: correlation_with_borrower_apr.png")
        print("Saved table: correlation_with_borrower_apr.csv")
        print(
            "Conclusion: Features with stronger correlation to BorrowerAPR may be "
            "useful for the regression task because BorrowerAPR reflects risk-based "
            "loan pricing."
        )

        print("========== 4.2.2 High Correlation Between Features ==========")
        threshold = 0.85
        high_corr_pairs = build_high_correlation_pairs(corr_matrix, threshold)
        print_table_from_pandas(high_corr_pairs)
        high_corr_pairs.to_csv(
            os.path.join(OUTPUT_DIR, "high_correlation_pairs.csv"),
            index=False,
        )
        print("Saved table: high_correlation_pairs.csv")
        print(
            "Conclusion: Highly correlated feature pairs suggest redundant information "
            "and support EDA-based feature reduction."
        )

        print("========== 4.2.3 Redundant Feature Detection ==========")
        redundant_remove_features = set()
        target_correlations = corr_matrix[target_col].abs()

        for _, row in high_corr_pairs.iterrows():
            feature_1 = row["Feature_1"]
            feature_2 = row["Feature_2"]

            if feature_1 == target_col or feature_2 == target_col:
                continue

            corr_1 = target_correlations.get(feature_1, 0)
            corr_2 = target_correlations.get(feature_2, 0)

            if pd.isna(corr_1):
                corr_1 = 0
            if pd.isna(corr_2):
                corr_2 = 0

            if corr_1 >= corr_2:
                redundant_remove_features.add(feature_2)
            else:
                redundant_remove_features.add(feature_1)

        redundant_remove_features = sorted(redundant_remove_features)
        print_feature_list(
            "Redundant features suggested for removal",
            redundant_remove_features,
        )
        print(
            "Conclusion: When two predictors contain similar information, the feature "
            "with weaker relationship to BorrowerAPR is removed as redundant."
        )

        print("========== 4.2.4 Feature Reduction Based on Correlation Analysis ==========")
        correlation_removal_pd = pd.DataFrame(
            [
                {
                    "Feature": feature,
                    "Reason": (
                        "Highly correlated with another numerical feature and has "
                        "weaker correlation with BorrowerAPR."
                    ),
                }
                for feature in redundant_remove_features
            ],
            columns=["Feature", "Reason"],
        )
        print_table_from_pandas(correlation_removal_pd)
        for feature in redundant_remove_features:
            add_removal_record(
                removal_records,
                feature,
                "Correlation Redundancy Removal",
                (
                    "Highly correlated with another numerical feature and has weaker "
                    "correlation with BorrowerAPR."
                ),
            )
        print(
            "Conclusion: Correlation-based feature reduction removes redundant "
            "numerical variables while keeping the stronger risk-pricing signal."
        )

        print("========== 4.2.5 Correlation Matrix Heatmap ==========")
        plt.figure(figsize=(12, 10))
        sns.heatmap(
            corr_matrix,
            cmap="coolwarm",
            center=0,
            annot=False,
            linewidths=0.5,
        )
        plt.title("Correlation Matrix of Numerical Features")
        plt.tight_layout()
        plt.savefig(
            os.path.join(OUTPUT_DIR, "correlation_matrix_heatmap.png"),
            dpi=300,
            bbox_inches="tight",
        )
        plt.close()
        print("Saved chart: correlation_matrix_heatmap.png")
        print(
            "Conclusion: The heatmap provides a compact overview of numerical feature "
            "relationships and highlights areas of redundancy."
        )

    removed_features_by_stage_pd = pd.DataFrame(
        removal_records,
        columns=["RemovedFeature", "RemovalStage", "Reason"],
    ).drop_duplicates(subset=["RemovedFeature"], keep="first")
    removed_features_by_stage_pd.to_csv(
        os.path.join(OUTPUT_DIR, "removed_features_by_eda.csv"),
        index=False,
    )

    final_remove_features = removed_features_by_stage_pd["RemovedFeature"].tolist()
    df_eda_reduced = drop_features_if_present(df, final_remove_features)

    print("========== EDA FEATURE REDUCTION SUMMARY ==========")
    print(f"Original columns before EDA reduction: {original_column_count}")
    print_feature_list(
        "Features removed in Chapter 4.1",
        distribution_removed_features,
    )
    print_feature_list(
        "Features removed in Chapter 4.2 manual removal",
        manual_remove_features,
    )
    print_feature_list(
        "Features removed in Chapter 4.2 correlation redundancy",
        redundant_remove_features,
    )
    print(f"Final columns after EDA reduction: {len(df_eda_reduced.columns)}")
    print(f"HDFS path of saved EDA-reduced dataset: {HDFS_EDA_REDUCED_DATASET_PATH}")
    print("Saved table: removed_features_by_eda.csv")

    print("========== SAVE EDA-REDUCED DATASET TO HDFS ==========")
    df_eda_reduced.write.mode("overwrite").parquet(HDFS_EDA_REDUCED_DATASET_PATH)
    print(f"Saved EDA-reduced dataset to: {HDFS_EDA_REDUCED_DATASET_PATH}")

    print("========== EDA COMPLETED ==========")
    spark.stop()


if __name__ == "__main__":
    main()


========== READ REDUCED DATASET ==========
Reduced dataset loaded from: hdfs://localhost:9000/bigdata/prosper_loan/processed/prosper_loan_reduced
========== DATASET SUMMARY ==========
Total rows: 113937
Total columns: 23
Columns:
- ListingCreationDate
- Term
- LoanStatus
- BorrowerAPR
- EstimatedLoss
- ProsperScore
- IsBorrowerHomeowner
- DateCreditPulled
- CreditScoreRangeLower
- FirstRecordedCreditLine
- InquiriesLast6Months
- CurrentDelinquencies
- DelinquenciesLast7Years
- BankcardUtilization
- TotalTrades
- DebtToIncomeRatio
- IncomeRange
- TotalProsperLoans
- ProsperPaymentsLessThanOneMonthLate
- ProsperPaymentsOneMonthPlusLate
- LoanOriginalAmount
- LoanOriginationDate
- Investors
========== 4.1 DISTRIBUTION ANALYSIS ==========
========== LOAN OUTCOME DISTRIBUTION ==========
        outcome_group  count  percentage
Other / Not Finalized  58853       51.65
            Good Loan  38074       33.42
             Bad Loan  17010       14.93
Conclusion: The dataset contains a large pr

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    when,
    count,
    isnan,
    sum as spark_sum,
    year,
    month,
    datediff,
    to_date,
    lit,
    percentile_approx,
    avg,
    min as spark_min,
    max as spark_max,
    stddev,
)
from pyspark.sql.types import DoubleType, IntegerType
import csv
import math
import os


# ============================================================
# FILE 05 OVERVIEW
# ============================================================
#
# Input:
# The script reads the EDA-reduced Prosper Loan dataset created by File 04:
# prosper_loan_eda_reduced
#
# Main tasks:
# 1. Read the cleaned feature set from HDFS.
# 2. Check and remove duplicate loan records.
# 3. Validate that important columns are still available.
# 4. Remove features that are too sparse to be reliable.
# 5. Review and handle missing values.
# 6. Create useful date-based features from raw date columns.
# 7. Standardize numeric and categorical data types.
# 8. Create a classification dataset for Good Loan / Bad Loan prediction.
# 9. Create a regression dataset for BorrowerAPR prediction.
# 10. Save the preprocessed outputs back to HDFS.
#
# Why this file exists:
# File 04 focuses on EDA-based feature reduction. This file prepares the data
# for later Feature Selection and Machine Learning Modeling by cleaning values,
# creating targets, and saving task-specific datasets.
#
# Outputs:
# prosper_loan_preprocessed_classification
# prosper_loan_preprocessed_regression
#
# These outputs are intended for File 06 Feature Selection and later Spark ML
# modeling scripts.
# ============================================================


HDFS_INPUT_PATH = "hdfs://localhost:9000/bigdata/prosper_loan/processed/prosper_loan_eda_reduced"
HDFS_CLASSIFICATION_OUTPUT_PATH = (
    "hdfs://localhost:9000/bigdata/prosper_loan/processed/"
    "prosper_loan_preprocessed_classification"
)
HDFS_REGRESSION_OUTPUT_PATH = (
    "hdfs://localhost:9000/bigdata/prosper_loan/processed/"
    "prosper_loan_preprocessed_regression"
)

LOCAL_TABLE_OUTPUT_DIR = os.path.join("outputs", "tables")
MISSING_VALUE_SUMMARY_PATH = os.path.join(
    LOCAL_TABLE_OUTPUT_DIR,
    "missing_value_summary.csv",
)


NUMERIC_COLUMNS = [
    "Term",
    "BorrowerAPR",
    "EstimatedLoss",
    "ProsperScore",
    "CreditScoreRangeLower",
    "InquiriesLast6Months",
    "CurrentDelinquencies",
    "DelinquenciesLast7Years",
    "BankcardUtilization",
    "TotalTrades",
    "DebtToIncomeRatio",
    "TotalProsperLoans",
    "ProsperPaymentsLessThanOneMonthLate",
    "ProsperPaymentsOneMonthPlusLate",
    "LoanOriginalAmount",
    "Investors",
    "listing_year",
    "listing_month",
    "loan_origination_year",
    "loan_origination_month",
    "credit_history_days",
    "credit_pull_to_origination_days",
]

CATEGORICAL_COLUMNS = [
    "IncomeRange",
    "EmploymentStatus",
    "IsBorrowerHomeowner",
]

RAW_DATE_COLUMNS = [
    "ListingCreationDate",
    "LoanOriginationDate",
    "DateCreditPulled",
    "FirstRecordedCreditLine",
]

HIGH_MISSING_VALUE_COLUMNS = [
    "TotalProsperLoans",
    "ProsperPaymentsLessThanOneMonthLate",
    "ProsperPaymentsOneMonthPlusLate",
]


def print_header(title):
    print(f"\n========== {title} ==========")


def print_status(title, df):
    print_header(title)
    print(f"Rows: {df.count()}")
    print(f"Columns: {len(df.columns)}")
    print("Column list:")
    for index, column_name in enumerate(df.columns, start=1):
        print(f"{index:02d}. {column_name}")


def get_existing_cols(df, cols):
    return [column_name for column_name in cols if column_name in df.columns]


def safe_drop(df, cols):
    existing_cols = get_existing_cols(df, cols)
    if not existing_cols:
        return df
    return df.drop(*existing_cols)


def remove_duplicate_rows(df):
    print_header("DUPLICATE CHECKING AND REMOVAL")

    # Duplicate removal ensures each loan record is counted only once and prevents
    # repeated rows from biasing downstream model training.
    # Count rows before removing duplicates so the cleaning effect is measurable.
    before_duplicate_count = df.count()

    # Remove duplicate records across all columns. This keeps one copy of each
    # identical loan row and prevents the same information from being learned twice.
    df = df.dropDuplicates()

    # Count rows after cleaning to show how many duplicated records were removed.
    after_duplicate_count = df.count()
    duplicate_removed_count = before_duplicate_count - after_duplicate_count

    print(f"Row count before duplicate removal: {before_duplicate_count}")
    print(f"Row count after duplicate removal: {after_duplicate_count}")
    print(f"Duplicate rows removed: {duplicate_removed_count}")

    return df


def drop_high_missing_value_columns(df):
    print_header("HIGH-MISSING-VALUE FEATURE REMOVAL")
    print(
        "Columns with missing value percentages above 80% are removed because they "
        "are unreliable and may introduce noise into the machine learning models."
    )

    columns_before = len(df.columns)

    # File 04 may already have removed these columns. This check prevents errors
    # by dropping only the features that still exist in the current DataFrame.
    existing_drop_cols = get_existing_cols(df, HIGH_MISSING_VALUE_COLUMNS)

    if existing_drop_cols:
        print("High-missing-value columns removed:")
        for column_name in existing_drop_cols:
            print(f"- {column_name}")
        df = df.drop(*existing_drop_cols)
    else:
        print(
            "No configured high-missing-value columns were found. They may have "
            "already been removed by File 04 EDA-based feature reduction."
        )

    print(f"Columns before removal: {columns_before}")
    print(f"Columns removed: {len(existing_drop_cols)}")
    print(f"Columns after removal: {len(df.columns)}")
    print("Remaining columns after high-missing-value cleanup:")
    for index, column_name in enumerate(df.columns, start=1):
        print(f"{index:02d}. {column_name}")

    return df


def validate_dataset_schema(df, stage_description):
    print_header("BASIC SCHEMA AND DATA VALIDATION")

    # Row and column counts help confirm that preprocessing did not accidentally
    # remove all records or remove too many features.
    print(f"Total row count {stage_description}: {df.count()}")
    print(f"Total column count {stage_description}: {len(df.columns)}")

    # LoanStatus is needed for classification. BorrowerAPR is needed for regression.
    # Checking them early makes pipeline problems easier to diagnose.
    important_columns = ["LoanStatus", "BorrowerAPR"]
    print("Important columns before target construction:")
    for column_name in important_columns:
        status = "present" if column_name in df.columns else "missing"
        print(f"- {column_name}: {status}")


def calculate_missing_summary(df):
    # Missing-value analysis shows which features may need imputation or removal.
    # This script counts NULL values only. NaN handling for numeric values happens
    # later during numeric imputation.
    total_rows = df.count()
    agg_exprs = [
        spark_sum(when(col(column_name).isNull(), 1).otherwise(0)).alias(column_name)
        for column_name in df.columns
    ]

    missing_counts = df.select(agg_exprs).collect()[0].asDict()

    summary = []
    for column_name, missing_count in missing_counts.items():
        missing_count = int(missing_count or 0)
        missing_percent = 0.0
        if total_rows > 0:
            missing_percent = round((missing_count / total_rows) * 100, 2)
        summary.append(
            {
                "column": column_name,
                "missing_count": missing_count,
                "missing_percent": missing_percent,
            }
        )

    return sorted(summary, key=lambda row: row["missing_percent"], reverse=True)


def save_missing_summary_locally(missing_summary):
    os.makedirs(LOCAL_TABLE_OUTPUT_DIR, exist_ok=True)
    with open(MISSING_VALUE_SUMMARY_PATH, "w", newline="", encoding="utf-8") as output_file:
        writer = csv.DictWriter(
            output_file,
            fieldnames=["column", "missing_count", "missing_percent"],
        )
        writer.writeheader()
        writer.writerows(missing_summary)

    print(f"Missing value summary saved to: {MISSING_VALUE_SUMMARY_PATH}")


def create_date_features(df):
    created_features = []

    if "ListingCreationDate" in df.columns:
        # Listing year and month capture broad timing patterns without keeping
        # the full raw date string.
        df = df.withColumn(
            "listing_year",
            year(to_date(col("ListingCreationDate"))),
        ).withColumn(
            "listing_month",
            month(to_date(col("ListingCreationDate"))),
        )
        created_features.extend(["listing_year", "listing_month"])

    if "LoanOriginationDate" in df.columns:
        # Loan origination year and month can help later models learn changes in
        # lending behavior over time.
        df = df.withColumn(
            "loan_origination_year",
            year(to_date(col("LoanOriginationDate"))),
        ).withColumn(
            "loan_origination_month",
            month(to_date(col("LoanOriginationDate"))),
        )
        created_features.extend(["loan_origination_year", "loan_origination_month"])

    if "LoanOriginationDate" in df.columns and "FirstRecordedCreditLine" in df.columns:
        # Credit history length is a borrower risk signal. A longer credit history
        # often gives lenders more evidence about repayment behavior.
        df = df.withColumn(
            "credit_history_days",
            datediff(
                to_date(col("LoanOriginationDate")),
                to_date(col("FirstRecordedCreditLine")),
            ),
        )
        created_features.append("credit_history_days")

    if "LoanOriginationDate" in df.columns and "DateCreditPulled" in df.columns:
        # This feature measures the time between the credit pull and loan origination.
        # It can reveal whether the credit information was recent.
        df = df.withColumn(
            "credit_pull_to_origination_days",
            datediff(
                to_date(col("LoanOriginationDate")),
                to_date(col("DateCreditPulled")),
            ),
        )
        created_features.append("credit_pull_to_origination_days")

    # After extracting useful date features, raw date columns are removed so later
    # ML steps do not need to handle date strings directly.
    df = safe_drop(df, RAW_DATE_COLUMNS)

    print("Date features created from raw temporal columns.")
    if created_features:
        print("Created date features:")
        for feature_name in created_features:
            print(f"- {feature_name}")
    else:
        print("No expected raw temporal columns were found for date feature engineering.")

    return df


def standardize_data_types(df):
    existing_numeric_cols = get_existing_cols(df, NUMERIC_COLUMNS)
    existing_categorical_cols = get_existing_cols(df, CATEGORICAL_COLUMNS)

    for column_name in existing_numeric_cols:
        # Numeric columns are cast to DoubleType because Spark ML algorithms expect
        # numerical features to have consistent numeric types.
        df = df.withColumn(column_name, col(column_name).cast(DoubleType()))

    for column_name in existing_categorical_cols:
        if column_name == "IsBorrowerHomeowner":
            # Convert homeowner status into clear string categories so it can be
            # encoded later in the modeling stage.
            df = df.withColumn(
                column_name,
                when(col(column_name).isNull(), lit("Unknown"))
                .when(col(column_name).cast("boolean") == lit(True), lit("true"))
                .when(col(column_name).cast("boolean") == lit(False), lit("false"))
                .otherwise(col(column_name).cast("string")),
            )
        else:
            # Keep categorical variables as strings. Encoding is intentionally left
            # for the later ML modeling stage.
            df = df.withColumn(column_name, col(column_name).cast("string"))

    print("Numeric columns cast to DoubleType:")
    if existing_numeric_cols:
        for column_name in existing_numeric_cols:
            print(f"- {column_name}")
    else:
        print("- None found")

    print("Categorical columns retained for later encoding:")
    if existing_categorical_cols:
        for column_name in existing_categorical_cols:
            print(f"- {column_name}")
    else:
        print("- None found")

    if "IsBorrowerHomeowner" in existing_categorical_cols:
        print('IsBorrowerHomeowner standardized to string values: "true", "false", "Unknown".')

    return df


def get_numeric_fill_value(df, column_name):
    # Median imputation is used because it is less sensitive to extreme values
    # than the mean. This fills missing numeric values without changing the row count.
    valid_rows = df.filter(
        col(column_name).isNotNull() & (~isnan(col(column_name).cast("double")))
    )
    median_row = valid_rows.select(
        percentile_approx(col(column_name).cast("double"), 0.5).alias("median")
    ).collect()[0]
    fill_value = median_row["median"]

    if fill_value is None:
        return 0.0
    if isinstance(fill_value, float) and math.isnan(fill_value):
        return 0.0
    return float(fill_value)


def fill_numeric_with_median(df, numeric_cols):
    existing_numeric_cols = get_existing_cols(df, numeric_cols)

    if not existing_numeric_cols:
        print("No expected numeric columns found for median imputation.")
        return df

    for column_name in existing_numeric_cols:
        fill_value = get_numeric_fill_value(df, column_name)

        # Replace NULL and NaN values with the column median so Spark ML receives
        # complete numeric inputs later.
        df = df.withColumn(
            column_name,
            when(
                col(column_name).isNull() | isnan(col(column_name).cast("double")),
                lit(fill_value),
            )
            .otherwise(col(column_name))
            .cast(DoubleType()),
        )
        print(f"{column_name}: missing numeric values filled with {fill_value}")

    return df


def fill_categorical_with_unknown(df):
    existing_categorical_cols = get_existing_cols(df, CATEGORICAL_COLUMNS)

    if not existing_categorical_cols:
        print("No expected categorical columns found for Unknown imputation.")
        return df

    for column_name in existing_categorical_cols:
        # Unknown is used as an explicit category. This preserves rows while making
        # missing categorical values visible to later encoding steps.
        df = df.withColumn(
            column_name,
            when(col(column_name).isNull(), lit("Unknown"))
            .otherwise(col(column_name).cast("string")),
        )
        print(f"{column_name}: missing categorical values filled with Unknown")

    return df


def print_label_distribution(df):
    total_rows = df.count()
    label_rows = (
        df.groupBy("label")
        .agg(count("*").alias("count"))
        .orderBy("label")
        .collect()
    )

    print("label | count | percentage")
    print("--------------------------")
    for row in label_rows:
        percentage = 0.0
        if total_rows > 0:
            percentage = (row["count"] / total_rows) * 100
        label_name = "Good Loan" if row["label"] == 0 else "Bad Loan"
        print(f"{label_name} / label {row['label']} | {row['count']} | {percentage:.2f}%")


def build_classification_dataset(df_shared):
    if "LoanStatus" not in df_shared.columns:
        raise ValueError("LoanStatus is required to construct the classification label.")

    rows_before = df_shared.count()
    finalized_statuses = ["Completed", "Chargedoff", "Defaulted"]

    # Only finalized loans are used for classification because their final outcome
    # is known. Current loans are excluded because their repayment result could
    # still change in the future.
    finalized_count = df_shared.filter(col("LoanStatus").isin(finalized_statuses)).count()
    excluded_count = rows_before - finalized_count

    print("Classification status validation:")
    print(f"Rows with finalized statuses: {finalized_count}")
    print(f"Rows excluded because they are not finalized: {excluded_count}")

    df_classification = (
        df_shared.withColumn(
            "label",
            # Chargedoff and Defaulted loans are bad loans because they represent
            # credit risk events where the borrower did not successfully repay.
            when(col("LoanStatus").isin("Chargedoff", "Defaulted"), lit(1))
            # Completed loans are good loans because borrowers fully repaid their
            # obligations.
            .when(col("LoanStatus") == "Completed", lit(0))
            # Current and other non-finalized loans are not assigned a label because
            # their final outcome is still unknown.
            .otherwise(lit(None))
            .cast(IntegerType()),
        )
        .filter(col("label").isNotNull())
    )

    # BorrowerAPR may be removed in the classification ML file if treated as pricing leakage.
    df_classification = safe_drop(df_classification, ["LoanStatus"])

    print("Classification target mapping:")
    print("- Completed -> Good Loan -> label 0")
    print("- Chargedoff, Defaulted -> Bad Loan -> label 1")
    print("- Current and other non-finalized statuses are excluded.")
    print(f"Rows before finalized-status filtering: {rows_before}")
    print(f"Rows after finalized-status filtering: {df_classification.count()}")
    print_label_distribution(df_classification)
    print("Conclusion: Classification dataset uses only final loan outcomes to reduce label noise.")

    return df_classification


def print_borrower_apr_summary(df_regression):
    summary_row = (
        df_regression.select(
            count(col("BorrowerAPR")).alias("count"),
            avg(col("BorrowerAPR")).alias("mean"),
            spark_min(col("BorrowerAPR")).alias("min"),
            spark_max(col("BorrowerAPR")).alias("max"),
            stddev(col("BorrowerAPR")).alias("standard_deviation"),
        )
        .collect()[0]
    )

    print("BorrowerAPR summary:")
    print(f"count: {summary_row['count']}")
    print(f"mean: {summary_row['mean']}")
    print(f"min: {summary_row['min']}")
    print(f"max: {summary_row['max']}")
    print(f"standard deviation: {summary_row['standard_deviation']}")


def build_regression_dataset(df_shared):
    if "BorrowerAPR" not in df_shared.columns:
        raise ValueError("BorrowerAPR is required as the regression target.")

    rows_before = df_shared.count()

    # BorrowerAPR is the target variable for the regression task. Rows without APR
    # information cannot be used for training because the expected answer is missing.
    missing_borrower_apr_count = df_shared.filter(
        col("BorrowerAPR").isNull() | isnan(col("BorrowerAPR"))
    ).count()

    # Keep only rows where BorrowerAPR is available and valid.
    df_regression = df_shared.filter(
        col("BorrowerAPR").isNotNull() & (~isnan(col("BorrowerAPR")))
    )
    df_regression = safe_drop(df_regression, ["LoanStatus"])

    print(f"Rows with missing or NaN BorrowerAPR before filtering: {missing_borrower_apr_count}")
    print(f"Rows before BorrowerAPR filtering: {rows_before}")
    print(f"Rows after BorrowerAPR filtering: {df_regression.count()}")
    print_borrower_apr_summary(df_regression)
    print("Conclusion: Regression dataset keeps BorrowerAPR as the prediction target.")

    return df_regression


def main():
    spark = (
        SparkSession.builder
        .appName("Prosper Loan ML Preprocessing")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")

    # ============================================================
    # STEP 1: READ EDA-REDUCED DATASET FROM HDFS
    # ============================================================
    #
    # Purpose:
    # Read the dataset produced by File 04, which already completed EDA,
    # distribution analysis, correlation analysis, and EDA-based feature reduction.
    #
    # Why:
    # File 05 should not start from the earlier File 03 dataset because File 04
    # may have removed additional weak, redundant, or high-missing-value features.
    #
    # Output:
    # A Spark DataFrame containing the EDA-reduced loan records.
    #
    # Next step:
    # Check for duplicate rows so each loan record is counted only once.
    # ============================================================
    print_header("READ EDA-REDUCED DATASET")
    df = spark.read.parquet(HDFS_INPUT_PATH)
    print(f"EDA-reduced dataset loaded from: {HDFS_INPUT_PATH}")

    print_status("DATASET SUMMARY", df)

    # ============================================================
    # STEP 2: DUPLICATE CHECKING AND REMOVAL
    # ============================================================
    #
    # Purpose:
    # Detect and remove fully duplicated rows from the dataset.
    #
    # Why:
    # Duplicate loan records can make model training unfair because the same
    # borrower or loan pattern may be counted more than once.
    #
    # Output:
    # A DataFrame with duplicate rows removed, plus a console log showing how
    # many rows were removed.
    #
    # Next step:
    # Validate that important columns still exist after duplicate removal.
    # ============================================================
    df = remove_duplicate_rows(df)

    # ============================================================
    # STEP 3: BASIC SCHEMA AND DATA VALIDATION
    # ============================================================
    #
    # Purpose:
    # Confirm that the dataset still has rows, columns, and required target fields.
    #
    # Why:
    # LoanStatus is needed to build the classification target. BorrowerAPR is
    # needed to build the regression target. If either is missing, later outputs
    # cannot be created correctly.
    #
    # Output:
    # Console validation logs showing row count, column count, and whether key
    # target columns are present.
    #
    # Next step:
    # Remove any known unreliable features that may still exist after File 04.
    # ============================================================
    validate_dataset_schema(df, "after duplicate removal")

    # BorrowerRate is removed if it is still present because it is a direct pricing
    # proxy for BorrowerAPR and can leak information into downstream modeling.
    if "BorrowerRate" in df.columns:
        df = df.drop("BorrowerRate")
        print("BorrowerRate removed because it overlaps strongly with BorrowerAPR.")
        print(f"Columns after BorrowerRate removal: {len(df.columns)}")
    else:
        print("BorrowerRate was not present in the reduced dataset.")

    # ============================================================
    # STEP 4: HIGH-MISSING-VALUE FEATURE REMOVAL
    # ============================================================
    #
    # Purpose:
    # Remove specific Prosper-history columns if they still exist in the dataset.
    #
    # Why:
    # These columns were identified as having more than 80% missing values.
    # Features with this much missing information are less reliable and may add
    # noise to later machine learning models.
    #
    # Output:
    # A DataFrame without the configured high-missing-value columns, if those
    # columns are present.
    #
    # Next step:
    # Review the remaining missing values before imputation.
    # ============================================================
    df = drop_high_missing_value_columns(df)

    # ============================================================
    # STEP 5: MISSING VALUE ANALYSIS AND HANDLING
    # ============================================================
    #
    # Purpose:
    # Summarize missing values and later fill missing numeric and categorical
    # values in a consistent way.
    #
    # Why:
    # Spark ML models generally require complete feature values. Missing values
    # can cause model training errors or reduce model quality if handled poorly.
    #
    # Output:
    # A local missing-value summary table and, after type standardization, a
    # DataFrame with numeric NULL/NaN values filled and categorical NULL values
    # replaced by "Unknown".
    #
    # Next step:
    # Create date-based features before casting feature data types.
    # ============================================================
    print_header("MISSING VALUE ANALYSIS")
    missing_summary = calculate_missing_summary(df)
    print(f"{'column':35} {'missing_count':15} {'missing_percent':15}")
    for row in missing_summary[:30]:
        print(
            f"{row['column']:35} "
            f"{row['missing_count']:<15} "
            f"{row['missing_percent']:<15}"
        )
    save_missing_summary_locally(missing_summary)

    # ============================================================
    # STEP 6: DATE FEATURE ENGINEERING
    # ============================================================
    #
    # Purpose:
    # Convert useful raw date columns into numeric features such as year, month,
    # credit history length, and credit-pull timing.
    #
    # Why:
    # Raw date strings are difficult for ML models to use directly. Derived date
    # features turn time information into simpler numeric values.
    #
    # Output:
    # A shared DataFrame with new date features and without the original raw date
    # columns.
    #
    # Next step:
    # Standardize data types so numeric and categorical columns are consistent.
    # ============================================================
    print_header("SHARED DATE FEATURE ENGINEERING")
    df_shared = create_date_features(df)

    # ============================================================
    # STEP 7: DATA TYPE STANDARDIZATION
    # ============================================================
    #
    # Purpose:
    # Cast expected numeric columns to DoubleType and categorical columns to strings.
    #
    # Why:
    # Consistent data types make later feature selection and ML modeling easier.
    # Numeric columns must be numeric for calculations, while categorical columns
    # should remain strings until encoding is performed in a later modeling file.
    #
    # Output:
    # A shared preprocessed DataFrame with standardized feature types and filled
    # missing values.
    #
    # Next step:
    # Build task-specific datasets for classification and regression.
    # ============================================================
    print_header("SHARED DATA TYPE STANDARDIZATION")
    df_shared = standardize_data_types(df_shared)

    # Missing value handling is performed after type standardization so numeric
    # imputation and NaN checks operate on Spark numeric columns consistently.
    print_header("SHARED MISSING VALUE HANDLING")
    df_shared = fill_numeric_with_median(df_shared, NUMERIC_COLUMNS)
    df_shared = fill_categorical_with_unknown(df_shared)

    # Validate the shared preprocessed dataframe before target-specific outputs are built.
    validate_dataset_schema(df_shared, "after preprocessing")

    # ============================================================
    # STEP 8: CLASSIFICATION TARGET CONSTRUCTION
    # ============================================================
    #
    # Purpose:
    # Create the Good Loan / Bad Loan target variable for classification.
    #
    # Why:
    # Credit risk classification requires a clear final outcome. Completed loans
    # are successful repayments, while Chargedoff and Defaulted loans are credit
    # risk events.
    #
    # Output:
    # A classification DataFrame with label 0 for Good Loan and label 1 for Bad
    # Loan. Current and other non-finalized loans are excluded.
    #
    # Next step:
    # Build the regression dataset for BorrowerAPR prediction.
    # ============================================================
    print_header("BUILD CLASSIFICATION DATASET")
    df_classification = build_classification_dataset(df_shared)
    print(f"Classification rows: {df_classification.count()}")
    print(f"Classification columns: {len(df_classification.columns)}")

    # ============================================================
    # STEP 9: REGRESSION TARGET CONSTRUCTION
    # ============================================================
    #
    # Purpose:
    # Prepare the dataset for predicting BorrowerAPR.
    #
    # Why:
    # BorrowerAPR is a risk-pricing value. Rows without BorrowerAPR cannot be used
    # for supervised regression because the target value is missing.
    #
    # Output:
    # A regression DataFrame containing only rows with valid BorrowerAPR values.
    #
    # Next step:
    # Save both task-specific datasets to HDFS.
    # ============================================================
    print_header("BUILD REGRESSION DATASET")
    df_regression = build_regression_dataset(df_shared)
    print(f"Regression rows: {df_regression.count()}")
    print(f"Regression columns: {len(df_regression.columns)}")

    # ============================================================
    # STEP 10: SAVE PREPROCESSED DATASETS TO HDFS
    # ============================================================
    #
    # Purpose:
    # Save the classification and regression datasets for the next stages of the
    # project pipeline.
    #
    # Why:
    # HDFS storage allows later Spark jobs to read these prepared datasets without
    # repeating all preprocessing steps.
    #
    # Output:
    # prosper_loan_preprocessed_classification
    # prosper_loan_preprocessed_regression
    #
    # Next step:
    # File 06 can perform feature selection, and later files can train Spark ML
    # models using these outputs.
    # ============================================================
    print_header("SAVE PREPROCESSED DATASETS")
    df_classification.write.mode("overwrite").parquet(HDFS_CLASSIFICATION_OUTPUT_PATH)
    df_regression.write.mode("overwrite").parquet(HDFS_REGRESSION_OUTPUT_PATH)

    print("Preprocessed datasets saved successfully.")
    print(f"Classification rows and columns: {df_classification.count()} rows, {len(df_classification.columns)} columns")
    print(f"Regression rows and columns: {df_regression.count()} rows, {len(df_regression.columns)} columns")
    print(f"Classification output path: {HDFS_CLASSIFICATION_OUTPUT_PATH}")
    print(f"Regression output path: {HDFS_REGRESSION_OUTPUT_PATH}")

    spark.stop()


if __name__ == "__main__":
    main()



========== READ EDA-REDUCED DATASET ==========
EDA-reduced dataset loaded from: hdfs://localhost:9000/bigdata/prosper_loan/processed/prosper_loan_eda_reduced

========== DATASET SUMMARY ==========
Rows: 113937
Columns: 19
Column list:
01. ListingCreationDate
02. Term
03. LoanStatus
04. BorrowerAPR
05. EstimatedLoss
06. ProsperScore
07. IsBorrowerHomeowner
08. DateCreditPulled
09. CreditScoreRangeLower
10. FirstRecordedCreditLine
11. InquiriesLast6Months
12. CurrentDelinquencies
13. DelinquenciesLast7Years
14. BankcardUtilization
15. TotalTrades
16. DebtToIncomeRatio
17. IncomeRange
18. LoanOriginalAmount
19. LoanOriginationDate

========== DUPLICATE CHECKING AND REMOVAL ==========
Row count before duplicate removal: 113937
Row count after duplicate removal: 113937
Duplicate rows removed: 0

========== BASIC SCHEMA AND DATA VALIDATION ==========
Total row count after duplicate removal: 113937
Total column count after duplicate removal: 19
Important columns before target construction:
-

In [24]:
import os
import csv
import math

import numpy as np

from pyspark import StorageLevel
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, monotonically_increasing_id, when
from pyspark.sql.types import StringType, BooleanType
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    Imputer,
    OneHotEncoder,
    QuantileDiscretizer,
    StandardScaler,
    StringIndexer,
    VectorAssembler,
)
from pyspark.ml.stat import ChiSquareTest
from pyspark.ml.classification import (
    GBTClassifier,
    LogisticRegression,
    RandomForestClassifier,
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)
from pyspark.ml.functions import vector_to_array


APP_NAME = "ProsperLoan_SparkML_Classification"
INPUT_PATH = "hdfs://localhost:9000/bigdata/prosper_loan/processed/prosper_loan_preprocessed_classification"
LABEL_COL = "label"
RANDOM_SEED = 42
TRAIN_RATIO = 0.8

OUTPUT_DIR = os.path.join("outputs", "06_spark_ml_classification")
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")
FIGURE_DIR = os.path.join(OUTPUT_DIR, "figures")

CORE_FEATURES = ["BorrowerAPR", "EstimatedLoss", "ProsperScore"]
NUM_BUCKETS_FOR_CHI_SQUARE = 10
RF_SELECTION_NUM_TREES = 80
RF_SELECTION_MAX_DEPTH = 5

MODEL_SPECS = {
    "Logistic Regression": {
        "estimator": LogisticRegression(
            labelCol=LABEL_COL,
            featuresCol="features",
            weightCol="class_weight",
            maxIter=80,
            regParam=0.01,
            elasticNetParam=0.0,
            threshold=0.5,
        ),
        "use_scaler": True,
    },
    "Random Forest": {
        "estimator": RandomForestClassifier(
            labelCol=LABEL_COL,
            featuresCol="features",
            weightCol="class_weight",
            numTrees=80,
            maxDepth=6,
            seed=RANDOM_SEED,
        ),
        "use_scaler": False,
    },
    "GBTClassifier": {
        "estimator": GBTClassifier(
            labelCol=LABEL_COL,
            featuresCol="features",
            weightCol="class_weight",
            maxIter=60,
            maxDepth=4,
            stepSize=0.05,
            seed=RANDOM_SEED,
        ),
        "use_scaler": False,
    },
}

THRESHOLDS = [
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
]


# -----------------------------
# Basic utilities
# -----------------------------

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        ensure_dir(parent)


def write_csv(rows, path):
    if not rows:
        print(f"No rows to write: {path}")
        return
    ensure_parent_dir(path)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print(f"Saved: {path}")


def print_section(title):
    print("\n" + "=" * 88)
    print(title)
    print("=" * 88)


def safe_divide(a, b):
    return a / b if b else 0.0


# -----------------------------
# Data preparation
# -----------------------------

def load_data(spark):
    df = spark.read.parquet(INPUT_PATH).persist(StorageLevel.MEMORY_AND_DISK)
    df.count()
    return df


def detect_feature_columns(df):
    numeric_cols = []
    categorical_cols = []

    for field in df.schema.fields:
        if field.name == LABEL_COL:
            continue
        if isinstance(field.dataType, (StringType, BooleanType)):
            categorical_cols.append(field.name)
        else:
            numeric_cols.append(field.name)

    return numeric_cols, categorical_cols


def print_dataset_summary(df):
    total = df.count()
    print_section("DATASET SUMMARY")
    print(f"Rows: {total}")
    print(f"Columns: {len(df.columns)}")

    rows = df.groupBy(LABEL_COL).agg(count("*").alias("count")).orderBy(LABEL_COL).collect()
    print("\nLabel distribution:")
    for r in rows:
        label = int(r[LABEL_COL])
        name = "Good Loan" if label == 0 else "Bad Loan"
        cnt = int(r["count"])
        print(f"label={label} ({name}): {cnt} ({safe_divide(cnt, total) * 100:.2f}%)")


def stratified_split(df):
    labels = [row[LABEL_COL] for row in df.select(LABEL_COL).distinct().collect()]
    fractions = {label: TRAIN_RATIO for label in labels}

    df_with_id = df.withColumn("_row_id", monotonically_increasing_id())
    train_ids = df_with_id.sampleBy(LABEL_COL, fractions=fractions, seed=RANDOM_SEED).select("_row_id")

    train_df = df_with_id.join(train_ids, on="_row_id", how="inner").drop("_row_id")
    test_df = df_with_id.join(train_ids, on="_row_id", how="left_anti").drop("_row_id")

    train_df = train_df.persist(StorageLevel.MEMORY_AND_DISK)
    test_df = test_df.persist(StorageLevel.MEMORY_AND_DISK)
    train_df.count()
    test_df.count()
    return train_df, test_df


def add_class_weight(df):
    label_counts = {
        int(row[LABEL_COL]): int(row["count"])
        for row in df.groupBy(LABEL_COL).agg(count("*").alias("count")).collect()
    }
    total = sum(label_counts.values())
    n_classes = len(label_counts)
    class_weights = {
        label: total / (n_classes * cnt)
        for label, cnt in label_counts.items()
    }

    weighted_df = df.withColumn(
        "class_weight",
        when(col(LABEL_COL) == 0, float(class_weights.get(0, 1.0)))
        .when(col(LABEL_COL) == 1, float(class_weights.get(1, 1.0)))
        .otherwise(1.0),
    )
    return weighted_df, class_weights


# -----------------------------
# Feature selection diagnostic
# -----------------------------

def negative_log10(p_value):
    if p_value is None:
        return 0.0
    p_value = float(p_value)
    if p_value <= 0:
        return 50.0
    return min(-math.log10(p_value), 50.0)


def normalize(score_dict):
    if not score_dict:
        return {}
    max_value = max(score_dict.values())
    if max_value <= 0:
        return {k: 0.0 for k in score_dict}
    return {k: v / max_value for k, v in score_dict.items()}


def chi_square_scores(train_df, numeric_cols, categorical_cols):
    numeric_bucket_cols = [f"{c}__bucket" for c in numeric_cols]
    categorical_index_cols = [f"{c}__idx_chi" for c in categorical_cols]
    stages = []

    if numeric_cols:
        stages.append(
            QuantileDiscretizer(
                inputCols=numeric_cols,
                outputCols=numeric_bucket_cols,
                numBuckets=NUM_BUCKETS_FOR_CHI_SQUARE,
                handleInvalid="keep",
            )
        )

    if categorical_cols:
        stages.append(
            StringIndexer(
                inputCols=categorical_cols,
                outputCols=categorical_index_cols,
                handleInvalid="keep",
            )
        )

    stages.append(
        VectorAssembler(
            inputCols=numeric_bucket_cols + categorical_index_cols,
            outputCol="chi_features",
            handleInvalid="keep",
        )
    )

    model = Pipeline(stages=stages).fit(train_df)
    chi_df = model.transform(train_df).select("chi_features", LABEL_COL)
    result = ChiSquareTest.test(chi_df, featuresCol="chi_features", labelCol=LABEL_COL).head()

    features = numeric_cols + categorical_cols
    p_values = list(result["pValues"])
    statistics = list(result["statistics"])

    rows = []
    scores = {}
    for i, feature in enumerate(features):
        score = negative_log10(float(p_values[i]))
        rows.append(
            {
                "feature": feature,
                "chi_square_statistic": float(statistics[i]),
                "chi_square_p_value": float(p_values[i]),
                "chi_square_score": score,
            }
        )
        scores[feature] = score

    return rows, scores


def rf_importance_scores(train_df, numeric_cols, categorical_cols):
    weighted_df, _ = add_class_weight(train_df)
    categorical_index_cols = [f"{c}__idx_rf" for c in categorical_cols]
    stages = []

    if categorical_cols:
        stages.append(
            StringIndexer(
                inputCols=categorical_cols,
                outputCols=categorical_index_cols,
                handleInvalid="keep",
            )
        )

    stages.append(
        VectorAssembler(
            inputCols=numeric_cols + categorical_index_cols,
            outputCol="rf_selection_features",
            handleInvalid="keep",
        )
    )

    stages.append(
        RandomForestClassifier(
            labelCol=LABEL_COL,
            featuresCol="rf_selection_features",
            weightCol="class_weight",
            numTrees=RF_SELECTION_NUM_TREES,
            maxDepth=RF_SELECTION_MAX_DEPTH,
            seed=RANDOM_SEED,
        )
    )

    model = Pipeline(stages=stages).fit(weighted_df)
    importances = list(model.stages[-1].featureImportances.toArray())
    features = numeric_cols + categorical_cols

    rows = []
    scores = {}
    for i, feature in enumerate(features):
        importance = float(importances[i])
        rows.append({"feature": feature, "rf_importance": importance})
        scores[feature] = importance

    return rows, scores


def feature_reason(feature):
    reasons = {
        "BorrowerAPR": "Core Prosper pricing signal kept for classification.",
        "EstimatedLoss": "Core Prosper risk signal.",
        "ProsperScore": "Internal Prosper risk score.",
        "CreditScoreRangeLower": "Borrower credit quality.",
        "InquiriesLast6Months": "Recent credit-seeking behavior.",
        "CurrentDelinquencies": "Current delinquency risk.",
        "DelinquenciesLast7Years": "Long-term delinquency history.",
        "BankcardUtilization": "Credit utilization pressure.",
        "TotalTrades": "Credit history depth.",
        "DebtToIncomeRatio": "Debt burden relative to income.",
        "IncomeRange": "Repayment capacity.",
        "LoanOriginalAmount": "Loan exposure size.",
        "Term": "Risk exposure period.",
        "credit_history_days": "Length of credit history.",
        "credit_pull_to_origination_days": "Operational timing signal.",
        "IsBorrowerHomeowner": "Borrower profile information.",
        "listing_year": "Loan vintage or period effect.",
        "listing_month": "Possible monthly period effect.",
        "loan_origination_year": "Loan vintage or period effect.",
        "loan_origination_month": "Possible monthly period effect.",
    }
    return reasons.get(feature, "Retained after diagnostic feature scoring.")


def run_feature_diagnostic(train_df, numeric_cols, categorical_cols):
    print_section("FEATURE SELECTION DIAGNOSTIC")
    all_features = numeric_cols + categorical_cols

    chi_rows, chi_scores = chi_square_scores(train_df, numeric_cols, categorical_cols)
    rf_rows, rf_scores = rf_importance_scores(train_df, numeric_cols, categorical_cols)

    chi_norm = normalize(chi_scores)
    rf_norm = normalize(rf_scores)

    score_rows = []
    for feature in all_features:
        combined = 0.5 * chi_norm.get(feature, 0.0) + 0.5 * rf_norm.get(feature, 0.0)
        score_rows.append(
            {
                "feature": feature,
                "chi_square_score": float(chi_scores.get(feature, 0.0)),
                "chi_square_score_normalized": float(chi_norm.get(feature, 0.0)),
                "rf_importance": float(rf_scores.get(feature, 0.0)),
                "rf_importance_normalized": float(rf_norm.get(feature, 0.0)),
                "combined_score": float(combined),
                "selection_decision": "KEEP",
                "decision_reason": feature_reason(feature),
            }
        )

    score_rows = sorted(score_rows, key=lambda r: r["combined_score"], reverse=True)
    for i, row in enumerate(score_rows, start=1):
        row["rank"] = i

    selected_features = [r["feature"] for r in score_rows]
    removed_features = []

    print(f"Candidate features: {len(all_features)}")
    print(f"Selected features : {len(selected_features)}")
    print(f"Removed features  : {len(removed_features)}")
    print("Top 10 features by combined score:")
    for row in score_rows[:10]:
        print(f"{row['rank']:>2}. {row['feature']:<32} {row['combined_score']:.4f}")

    return selected_features, removed_features, score_rows


# -----------------------------
# Modeling
# -----------------------------

def preprocessing_stages(numeric_cols, categorical_cols, use_scaler=False):
    stages = []
    numeric_imputed_cols = [f"{c}__imputed" for c in numeric_cols]
    categorical_index_cols = [f"{c}__idx" for c in categorical_cols]
    categorical_ohe_cols = [f"{c}__ohe" for c in categorical_cols]

    if numeric_cols:
        stages.append(Imputer(inputCols=numeric_cols, outputCols=numeric_imputed_cols))

    if categorical_cols:
        stages.append(
            StringIndexer(
                inputCols=categorical_cols,
                outputCols=categorical_index_cols,
                handleInvalid="keep",
            )
        )
        stages.append(
            OneHotEncoder(
                inputCols=categorical_index_cols,
                outputCols=categorical_ohe_cols,
                handleInvalid="keep",
                dropLast=False,
            )
        )

    stages.append(
        VectorAssembler(
            inputCols=numeric_imputed_cols + categorical_ohe_cols,
            outputCol="raw_features",
            handleInvalid="keep",
        )
    )

    if use_scaler:
        stages.append(
            StandardScaler(
                inputCol="raw_features",
                outputCol="features",
                withStd=True,
                withMean=False,
            )
        )
    else:
        stages.append(VectorAssembler(inputCols=["raw_features"], outputCol="features"))

    return stages


def evaluate_predictions(predictions, model_name):
    accuracy = MulticlassClassificationEvaluator(
        labelCol=LABEL_COL, predictionCol="prediction", metricName="accuracy"
    ).evaluate(predictions)
    f1 = MulticlassClassificationEvaluator(
        labelCol=LABEL_COL, predictionCol="prediction", metricName="f1"
    ).evaluate(predictions)
    precision = MulticlassClassificationEvaluator(
        labelCol=LABEL_COL, predictionCol="prediction", metricName="weightedPrecision"
    ).evaluate(predictions)
    recall = MulticlassClassificationEvaluator(
        labelCol=LABEL_COL, predictionCol="prediction", metricName="weightedRecall"
    ).evaluate(predictions)
    roc_auc = BinaryClassificationEvaluator(
        labelCol=LABEL_COL, rawPredictionCol="rawPrediction", metricName="areaUnderROC"
    ).evaluate(predictions)
    pr_auc = BinaryClassificationEvaluator(
        labelCol=LABEL_COL, rawPredictionCol="rawPrediction", metricName="areaUnderPR"
    ).evaluate(predictions)

    tp = predictions.filter((col(LABEL_COL) == 1) & (col("prediction") == 1)).count()
    tn = predictions.filter((col(LABEL_COL) == 0) & (col("prediction") == 0)).count()
    fp = predictions.filter((col(LABEL_COL) == 0) & (col("prediction") == 1)).count()
    fn = predictions.filter((col(LABEL_COL) == 1) & (col("prediction") == 0)).count()

    bad_precision = safe_divide(tp, tp + fp)
    bad_recall = safe_divide(tp, tp + fn)
    bad_f1 = safe_divide(2 * bad_precision * bad_recall, bad_precision + bad_recall)
    good_recall = safe_divide(tn, tn + fp)
    balanced_accuracy = 0.5 * (bad_recall + good_recall)

    return {
        "model": model_name,
        "accuracy": float(accuracy),
        "f1": float(f1),
        "weighted_precision": float(precision),
        "weighted_recall": float(recall),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "balanced_accuracy": float(balanced_accuracy),
        "bad_loan_precision": float(bad_precision),
        "bad_loan_recall": float(bad_recall),
        "bad_loan_f1": float(bad_f1),
        "tp": int(tp),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
    }


def train_and_evaluate(train_df, test_df, numeric_cols, categorical_cols):
    print_section("MODEL TRAINING")
    weighted_train_df, class_weights = add_class_weight(train_df)

    print("Class weights:")
    for label, weight in sorted(class_weights.items()):
        name = "Good Loan" if label == 0 else "Bad Loan"
        print(f"label={label} ({name}) -> {weight:.6f}")

    model_objects = {}
    prediction_objects = {}
    result_rows = []

    for model_name, spec in MODEL_SPECS.items():
        pipeline = Pipeline(
            stages=preprocessing_stages(numeric_cols, categorical_cols, spec["use_scaler"])
            + [spec["estimator"]]
        )
        model = pipeline.fit(weighted_train_df)
        predictions = model.transform(test_df).persist(StorageLevel.MEMORY_AND_DISK)
        predictions.count()

        result = evaluate_predictions(predictions, model_name)
        model_objects[model_name] = model
        prediction_objects[model_name] = predictions
        result_rows.append(result)

        print(
            f"{model_name:<22} "
            f"Acc={result['accuracy']:.4f} "
            f"PR-AUC={result['pr_auc']:.4f} "
            f"BadRecall={result['bad_loan_recall']:.4f} "
            f"BadF1={result['bad_loan_f1']:.4f}"
        )

    result_rows = sorted(
        result_rows,
        key=lambda r: (r["pr_auc"], r["bad_loan_recall"], r["bad_loan_f1"]),
        reverse=True,
    )
    best_model_name = result_rows[0]["model"]
    print(f"Best model: {best_model_name}")

    return model_objects, prediction_objects, result_rows, best_model_name


# -----------------------------
# Threshold analysis
# -----------------------------

def predictions_with_bad_probability(predictions):
    return (
        predictions.withColumn("probability_array", vector_to_array(col("probability")))
        .withColumn("bad_loan_probability", col("probability_array")[1])
        .select(LABEL_COL, "bad_loan_probability")
        .persist(StorageLevel.MEMORY_AND_DISK)
    )


def evaluate_threshold(predictions, threshold):
    threshold_predictions = predictions.withColumn(
        "threshold_prediction",
        when(col("bad_loan_probability") >= threshold, 1.0).otherwise(0.0),
    )

    tp = threshold_predictions.filter((col(LABEL_COL) == 1) & (col("threshold_prediction") == 1)).count()
    tn = threshold_predictions.filter((col(LABEL_COL) == 0) & (col("threshold_prediction") == 0)).count()
    fp = threshold_predictions.filter((col(LABEL_COL) == 0) & (col("threshold_prediction") == 1)).count()
    fn = threshold_predictions.filter((col(LABEL_COL) == 1) & (col("threshold_prediction") == 0)).count()
    total = tp + tn + fp + fn

    bad_precision = safe_divide(tp, tp + fp)
    bad_recall = safe_divide(tp, tp + fn)
    bad_f1 = safe_divide(2 * bad_precision * bad_recall, bad_precision + bad_recall)
    good_recall = safe_divide(tn, tn + fp)

    return {
        "threshold": threshold,
        "accuracy": safe_divide(tp + tn, total),
        "balanced_accuracy": 0.5 * (bad_recall + good_recall),
        "bad_loan_precision": bad_precision,
        "bad_loan_recall": bad_recall,
        "bad_loan_f1": bad_f1,
        "good_loan_recall": good_recall,
        "false_positive_rate": safe_divide(fp, fp + tn),
        "false_negative_rate": safe_divide(fn, fn + tp),
        "tp": int(tp),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
    }


def run_threshold_analysis(best_predictions):
    print_section("THRESHOLD ANALYSIS")
    prob_df = predictions_with_bad_probability(best_predictions)
    prob_df.count()

    rows = [evaluate_threshold(prob_df, t) for t in THRESHOLDS]
    best_f1 = max(rows, key=lambda r: r["bad_loan_f1"])
    best_balanced = max(rows, key=lambda r: r["balanced_accuracy"])
    candidates = [r for r in rows if r["bad_loan_recall"] >= 0.70]
    recommended = max(candidates, key=lambda r: r["bad_loan_f1"]) if candidates else best_f1

    for row in rows:
        print(
            f"t={row['threshold']:.2f} "
            f"Acc={row['accuracy']:.4f} "
            f"BalAcc={row['balanced_accuracy']:.4f} "
            f"BadP={row['bad_loan_precision']:.4f} "
            f"BadR={row['bad_loan_recall']:.4f} "
            f"BadF1={row['bad_loan_f1']:.4f}"
        )

    print(
        f"Recommended threshold: {recommended['threshold']:.2f} "
        f"(BadF1={recommended['bad_loan_f1']:.4f}, "
        f"BadRecall={recommended['bad_loan_recall']:.4f})"
    )

    return rows, recommended, best_f1, best_balanced


# -----------------------------
# Visualizations
# -----------------------------

def plot_feature_scores(rows, path):
    import matplotlib.pyplot as plt

    ensure_parent_dir(path)
    rows = sorted(rows, key=lambda r: r["combined_score"], reverse=True)
    features = [r["feature"] for r in rows][::-1]
    scores = [r["combined_score"] for r in rows][::-1]

    plt.figure(figsize=(11, 8))
    plt.barh(features, scores, color="#3B82F6")
    plt.xlabel("Combined feature score")
    plt.title("Hybrid Feature Selection Diagnostic")
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()
    print(f"Saved: {path}")


def plot_model_comparison(rows, path):
    import matplotlib.pyplot as plt

    ensure_parent_dir(path)
    rows = list(reversed(rows))
    models = [r["model"] for r in rows]
    metrics = ["accuracy", "balanced_accuracy", "roc_auc", "pr_auc", "bad_loan_recall", "bad_loan_f1"]
    labels = ["Accuracy", "Balanced Acc.", "ROC-AUC", "PR-AUC", "Bad Recall", "Bad F1"]

    x = np.arange(len(models))
    width = 0.12
    colors = ["#94A3B8", "#60A5FA", "#34D399", "#FBBF24", "#F97316", "#8B5CF6"]

    plt.figure(figsize=(12, 6))
    for i, metric in enumerate(metrics):
        values = [r[metric] for r in rows]
        plt.bar(x + (i - len(metrics) / 2) * width + width / 2, values, width, label=labels[i], color=colors[i])

    plt.xticks(x, models)
    plt.ylim(0, 1)
    plt.ylabel("Metric value")
    plt.title("Classification Model Performance")
    plt.legend(ncol=3)
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()
    print(f"Saved: {path}")


def plot_threshold_tradeoff(rows, recommended_threshold, path):
    import matplotlib.pyplot as plt

    ensure_parent_dir(path)
    thresholds = [r["threshold"] for r in rows]

    plt.figure(figsize=(11, 6))
    plt.plot(thresholds, [r["accuracy"] for r in rows], marker="o", label="Accuracy", color="#64748B")
    plt.plot(thresholds, [r["balanced_accuracy"] for r in rows], marker="o", label="Balanced Accuracy", color="#2563EB")
    plt.plot(thresholds, [r["bad_loan_precision"] for r in rows], marker="o", label="Bad Loan Precision", color="#059669")
    plt.plot(thresholds, [r["bad_loan_recall"] for r in rows], marker="o", label="Bad Loan Recall", color="#DC2626")
    plt.plot(thresholds, [r["bad_loan_f1"] for r in rows], marker="o", label="Bad Loan F1", color="#7C3AED")
    plt.axvline(recommended_threshold, linestyle="--", color="#111827", label=f"Recommended = {recommended_threshold:.2f}")

    plt.ylim(0, 1)
    plt.xlabel("Bad Loan probability threshold")
    plt.ylabel("Metric value")
    plt.title("Threshold Trade-off - GBTClassifier")
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()
    print(f"Saved: {path}")


def plot_confusion_matrix(row, path):
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap

    ensure_parent_dir(path)
    matrix = np.array([[row["tn"], row["fp"]], [row["fn"], row["tp"]]])
    cmap = LinearSegmentedColormap.from_list("credit_risk_matrix", ["#F8FAFC", "#93C5FD", "#1D4ED8", "#0F172A"])

    plt.figure(figsize=(7.5, 6.5))
    image = plt.imshow(matrix, cmap=cmap)
    plt.title(f"GBTClassifier Confusion Matrix - Threshold {row['threshold']:.2f}")
    plt.xlabel("Predicted label")
    plt.ylabel("Actual label")
    plt.xticks([0, 1], ["Good Loan", "Bad Loan"])
    plt.yticks([0, 1], ["Good Loan", "Bad Loan"])

    max_value = matrix.max()
    labels = [["TN", "FP"], ["FN", "TP"]]
    for i in range(2):
        for j in range(2):
            color = "white" if matrix[i, j] > max_value * 0.55 else "#0F172A"
            plt.text(j, i, f"{labels[i][j]}\n{matrix[i, j]:,}", ha="center", va="center", fontsize=14, fontweight="bold", color=color)

    plt.colorbar(image, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()
    print(f"Saved: {path}")


def expanded_feature_names(pipeline_model, numeric_cols, categorical_cols):
    names = list(numeric_cols)
    labels_by_col = {}

    for stage in pipeline_model.stages:
        if stage.__class__.__name__ == "StringIndexerModel":
            for col_name, labels in zip(categorical_cols, stage.labelsArray):
                labels_by_col[col_name] = list(labels) + ["__unknown__"]

    for col_name in categorical_cols:
        names.extend([f"{col_name}={label}" for label in labels_by_col.get(col_name, ["__unknown__"])])

    return names


def tree_feature_importance_rows(pipeline_model, numeric_cols, categorical_cols):
    importances = list(pipeline_model.stages[-1].featureImportances.toArray())
    names = expanded_feature_names(pipeline_model, numeric_cols, categorical_cols)

    rows = []
    for i, importance in enumerate(importances):
        feature = names[i] if i < len(names) else f"feature_{i}"
        rows.append({"feature": feature, "importance": float(importance)})

    return sorted(rows, key=lambda r: r["importance"], reverse=True)


def plot_feature_importance(rows, path, top_n=15):
    import matplotlib.pyplot as plt

    ensure_parent_dir(path)
    top_rows = rows[:top_n]
    features = [r["feature"] for r in top_rows][::-1]
    values = [r["importance"] for r in top_rows][::-1]

    plt.figure(figsize=(11, 7))
    plt.barh(features, values, color="#2563EB")
    plt.xlabel("Feature importance")
    plt.title(f"Top {top_n} Feature Importances - GBTClassifier")
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()
    print(f"Saved: {path}")


# -----------------------------
# Main flow
# -----------------------------

def main():
    ensure_dir(TABLE_DIR)
    ensure_dir(FIGURE_DIR)

    spark = SparkSession.builder.appName(APP_NAME).getOrCreate()
    spark.sparkContext.setLogLevel("WARN")

    df = load_data(spark)
    if LABEL_COL not in df.columns:
        raise ValueError(f"Missing target column: {LABEL_COL}")

    print_dataset_summary(df)
    numeric_cols, categorical_cols = detect_feature_columns(df)

    for feature in CORE_FEATURES:
        if feature not in numeric_cols + categorical_cols:
            print(f"Warning: core feature not found: {feature}")

    train_df, test_df = stratified_split(df)
    print_section("TRAIN TEST SPLIT")
    print(f"Train rows: {train_df.count()}")
    print(f"Test rows : {test_df.count()}")

    selected_features, removed_features, feature_scores = run_feature_diagnostic(train_df, numeric_cols, categorical_cols)
    selected_numeric_cols = [f for f in selected_features if f in numeric_cols]
    selected_categorical_cols = [f for f in selected_features if f in categorical_cols]

    write_csv(feature_scores, os.path.join(TABLE_DIR, "06_feature_selection_scores.csv"))
    write_csv([r for r in feature_scores if r["selection_decision"] == "KEEP"], os.path.join(TABLE_DIR, "06_selected_features.csv"))
    plot_feature_scores(feature_scores, os.path.join(FIGURE_DIR, "06_feature_selection_scores.png"))

    model_objects, prediction_objects, model_results, best_model_name = train_and_evaluate(
        train_df,
        test_df,
        selected_numeric_cols,
        selected_categorical_cols,
    )

    write_csv(model_results, os.path.join(TABLE_DIR, "06_model_evaluation.csv"))
    plot_model_comparison(model_results, os.path.join(FIGURE_DIR, "06_model_comparison.png"))

    threshold_rows, recommended, best_f1, best_balanced = run_threshold_analysis(prediction_objects[best_model_name])
    write_csv(threshold_rows, os.path.join(TABLE_DIR, "06_threshold_analysis.csv"))
    plot_threshold_tradeoff(threshold_rows, recommended["threshold"], os.path.join(FIGURE_DIR, "06_threshold_tradeoff.png"))
    plot_confusion_matrix(recommended, os.path.join(FIGURE_DIR, "06_confusion_matrix_gbt.png"))

    if best_model_name == "GBTClassifier":
        importance_rows = tree_feature_importance_rows(
            model_objects[best_model_name],
            selected_numeric_cols,
            selected_categorical_cols,
        )
        write_csv(importance_rows, os.path.join(TABLE_DIR, "06_gbt_feature_importance.csv"))
        plot_feature_importance(importance_rows, os.path.join(FIGURE_DIR, "06_gbt_feature_importance.png"))

    print_section("FINAL SUMMARY")
    print(f"Best model: {best_model_name}")
    print(f"Recommended threshold: {recommended['threshold']:.2f}")
    print(f"Bad Loan Recall: {recommended['bad_loan_recall']:.4f}")
    print(f"Bad Loan F1: {recommended['bad_loan_f1']:.4f}")
    print(f"Balanced Accuracy: {recommended['balanced_accuracy']:.4f}")
    print(f"Tables: {TABLE_DIR}")
    print(f"Figures: {FIGURE_DIR}")

    spark.stop()


if __name__ == "__main__":
    main()



DATASET SUMMARY
Rows: 55084
Columns: 21

Label distribution:
label=0 (Good Loan): 38074 (69.12%)
label=1 (Bad Loan): 17010 (30.88%)

TRAIN TEST SPLIT
Train rows: 44214
Test rows : 10870

FEATURE SELECTION DIAGNOSTIC
Candidate features: 20
Selected features : 20
Removed features  : 0
Top 10 features by combined score:
 1. BorrowerAPR                      1.0000
 2. CreditScoreRangeLower            0.7816
 3. EstimatedLoss                    0.7500
 4. InquiriesLast6Months             0.6057
 5. loan_origination_year            0.5997
 6. ProsperScore                     0.5906
 7. CurrentDelinquencies             0.5815
 8. listing_year                     0.5690
 9. Term                             0.5449
10. DebtToIncomeRatio                0.5292
Saved: outputs\06_spark_ml_classification\tables\06_feature_selection_scores.csv
Saved: outputs\06_spark_ml_classification\tables\06_selected_features.csv
Saved: outputs\06_spark_ml_classification\figures\06_feature_selection_scores.png

MO

In [ ]:
import os
import csv
import math

from pyspark import StorageLevel
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    count,
    avg,
    min as spark_min,
    max as spark_max,
    stddev,
    abs as spark_abs,
    rand,
)
from pyspark.sql.types import StringType, BooleanType
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    Imputer,
    VectorAssembler,
    StandardScaler,
)
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator


INPUT_PATH = "hdfs://localhost:9000/bigdata/prosper_loan/processed/prosper_loan_preprocessed_regression"
MODEL_OUTPUT_PATH = "hdfs://localhost:9000/bigdata/prosper_loan/models/regression/best_borrower_apr_model"

TARGET_COL = "BorrowerAPR"
TRAIN_RATIO = 0.8
RANDOM_SEED = 42

OUTPUT_DIR = os.path.join("outputs", "07_spark_ml_regression")
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")
FIGURE_DIR = os.path.join(OUTPUT_DIR, "figures")

os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

DATASET_SUMMARY_CSV = os.path.join(TABLE_DIR, "07_regression_dataset_summary.csv")
FEATURE_SCORE_CSV = os.path.join(TABLE_DIR, "07_regression_feature_scores.csv")
MODEL_METRICS_CSV = os.path.join(TABLE_DIR, "07_regression_model_metrics.csv")
FEATURE_IMPORTANCE_CSV = os.path.join(TABLE_DIR, "07_gbt_feature_importance.csv")
PREDICTION_SAMPLE_CSV = os.path.join(TABLE_DIR, "07_best_model_prediction_sample.csv")

TARGET_DISTRIBUTION_FIG = os.path.join(FIGURE_DIR, "07_borrower_apr_distribution.png")
FEATURE_SCORE_FIG = os.path.join(FIGURE_DIR, "07_regression_feature_scores.png")
MODEL_COMPARISON_FIG = os.path.join(FIGURE_DIR, "07_regression_model_comparison.png")
ACTUAL_VS_PREDICTED_FIG = os.path.join(FIGURE_DIR, "07_actual_vs_predicted_gbt.png")
RESIDUAL_DISTRIBUTION_FIG = os.path.join(FIGURE_DIR, "07_residual_distribution_gbt.png")
FEATURE_IMPORTANCE_FIG = os.path.join(FIGURE_DIR, "07_gbt_feature_importance.png")


def print_header(title):
    print("\n" + "=" * 86)
    print(title)
    print("=" * 86)


def ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)


def write_dicts_to_csv(rows, path):
    if not rows:
        return
    ensure_parent_dir(path)
    fieldnames = list(rows[0].keys())
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"Saved table: {path}")


def detect_feature_columns(df, target_col=TARGET_COL):
    numeric_cols = []
    categorical_cols = []

    for field in df.schema.fields:
        name = field.name
        if name == target_col:
            continue
        if isinstance(field.dataType, (StringType, BooleanType)):
            categorical_cols.append(name)
        else:
            numeric_cols.append(name)

    return numeric_cols, categorical_cols


def summarize_dataset(df):
    row = df.select(
        count(col(TARGET_COL)).alias("rows"),
        avg(col(TARGET_COL)).alias("mean"),
        spark_min(col(TARGET_COL)).alias("min"),
        spark_max(col(TARGET_COL)).alias("max"),
        stddev(col(TARGET_COL)).alias("stddev"),
    ).collect()[0]

    summary = {
        "rows": int(row["rows"]),
        "columns": len(df.columns),
        "target": TARGET_COL,
        "target_mean": float(row["mean"]),
        "target_min": float(row["min"]),
        "target_max": float(row["max"]),
        "target_stddev": float(row["stddev"]),
    }

    write_dicts_to_csv([summary], DATASET_SUMMARY_CSV)
    return summary


def plot_target_distribution(df, output_path, sample_size=20000):
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib is not installed. Skip target distribution figure.")
        return

    values = [
        float(row[TARGET_COL])
        for row in df.select(TARGET_COL).orderBy(rand(seed=RANDOM_SEED)).limit(sample_size).collect()
    ]

    plt.figure(figsize=(10, 6))
    plt.hist(values, bins=40, edgecolor="white", color="#2F80ED")
    plt.xlabel("BorrowerAPR")
    plt.ylabel("Frequency")
    plt.title("Distribution of BorrowerAPR")
    plt.tight_layout()
    plt.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.close()
    print(f"Saved figure: {output_path}")


def train_test_split(df):
    train_df, test_df = df.randomSplit([TRAIN_RATIO, 1 - TRAIN_RATIO], seed=RANDOM_SEED)
    return (
        train_df.persist(StorageLevel.MEMORY_AND_DISK),
        test_df.persist(StorageLevel.MEMORY_AND_DISK),
    )


def minmax_normalize(scores):
    if not scores:
        return {}
    max_value = max(scores.values())
    if max_value <= 0:
        return {key: 0.0 for key in scores}
    return {key: value / max_value for key, value in scores.items()}


def correlation_scores(train_df, numeric_cols):
    rows = []
    scores = {}

    for c in numeric_cols:
        try:
            corr_value = train_df.stat.corr(c, TARGET_COL)
        except Exception:
            corr_value = None

        if corr_value is None or math.isnan(corr_value):
            corr_value = 0.0

        abs_corr = abs(float(corr_value))
        rows.append({
            "feature": c,
            "pearson_correlation": float(corr_value),
            "absolute_correlation": abs_corr,
        })
        scores[c] = abs_corr

    return rows, scores


def rf_regression_importance_scores(train_df, numeric_cols, categorical_cols):
    indexed_cols = [f"{c}__idx_rf" for c in categorical_cols]
    stages = []

    if categorical_cols:
        stages.append(StringIndexer(
            inputCols=categorical_cols,
            outputCols=indexed_cols,
            handleInvalid="keep",
        ))

    feature_cols = numeric_cols + indexed_cols
    stages.append(VectorAssembler(
        inputCols=feature_cols,
        outputCol="rf_features",
        handleInvalid="keep",
    ))

    rf = RandomForestRegressor(
        labelCol=TARGET_COL,
        featuresCol="rf_features",
        numTrees=80,
        maxDepth=6,
        seed=RANDOM_SEED,
    )
    stages.append(rf)

    model = Pipeline(stages=stages).fit(train_df)
    rf_model = model.stages[-1]
    importances = list(rf_model.featureImportances.toArray())

    raw_features = numeric_cols + categorical_cols
    rows = []
    scores = {}

    for i, feature in enumerate(raw_features):
        importance = float(importances[i]) if i < len(importances) else 0.0
        rows.append({"feature": feature, "rf_importance": importance})
        scores[feature] = importance

    return rows, scores


def regression_feature_reason(feature):
    reasons = {
        "EstimatedLoss": "Strong Prosper risk signal closely related to loan pricing.",
        "ProsperScore": "Internal Prosper risk score reflecting borrower risk level.",
        "CreditScoreRangeLower": "Borrower credit quality indicator.",
        "LoanOriginalAmount": "Loan exposure size that may affect pricing.",
        "BankcardUtilization": "Credit utilization pressure.",
        "DelinquenciesLast7Years": "Long-term delinquency history.",
        "CurrentDelinquencies": "Current delinquency risk.",
        "InquiriesLast6Months": "Recent credit-seeking behavior.",
        "DebtToIncomeRatio": "Debt burden relative to income.",
        "TotalTrades": "Credit history depth.",
        "Term": "Loan duration and risk exposure period.",
        "listing_year": "Loan vintage or period effect.",
        "listing_month": "Possible monthly period effect.",
        "loan_origination_year": "Loan origination period effect.",
        "loan_origination_month": "Possible origination month effect.",
        "credit_history_days": "Length of borrower credit history.",
        "credit_pull_to_origination_days": "Operational timing signal.",
        "IncomeRange": "Borrower repayment capacity.",
        "IsBorrowerHomeowner": "Borrower profile information.",
    }
    return reasons.get(feature, "Retained after regression feature diagnostic.")


def regression_feature_diagnostic(train_df, numeric_cols, categorical_cols):
    corr_rows, corr_scores = correlation_scores(train_df, numeric_cols)
    _, rf_scores = rf_regression_importance_scores(train_df, numeric_cols, categorical_cols)

    corr_norm = minmax_normalize(corr_scores)
    rf_norm = minmax_normalize(rf_scores)

    all_features = numeric_cols + categorical_cols
    rows = []

    for feature in all_features:
        combined_score = 0.5 * corr_norm.get(feature, 0.0) + 0.5 * rf_norm.get(feature, 0.0)
        rows.append({
            "feature": feature,
            "absolute_correlation": corr_scores.get(feature, 0.0),
            "correlation_score_normalized": corr_norm.get(feature, 0.0),
            "rf_importance": rf_scores.get(feature, 0.0),
            "rf_importance_normalized": rf_norm.get(feature, 0.0),
            "combined_score": combined_score,
            "selection_decision": "KEEP",
            "decision_reason": regression_feature_reason(feature),
        })

    rows = sorted(rows, key=lambda row: row["combined_score"], reverse=True)
    for rank, row in enumerate(rows, start=1):
        row["rank"] = rank

    write_dicts_to_csv(rows, FEATURE_SCORE_CSV)
    return rows


def plot_feature_scores(rows, output_path, top_n=20):
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib is not installed. Skip feature score figure.")
        return

    top_rows = rows[:top_n]
    features = [r["feature"] for r in top_rows][::-1]
    scores = [r["combined_score"] for r in top_rows][::-1]

    plt.figure(figsize=(11, 7))
    plt.barh(features, scores, color="#2F80ED")
    plt.xlabel("Combined feature score")
    plt.ylabel("Feature")
    plt.title("Regression Feature Diagnostic for BorrowerAPR")
    plt.tight_layout()
    plt.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.close()
    print(f"Saved figure: {output_path}")


def build_preprocessing_stages(numeric_cols, categorical_cols, use_scaler=False):
    stages = []
    numeric_imputed_cols = [f"{c}__imputed" for c in numeric_cols]
    categorical_index_cols = [f"{c}__idx" for c in categorical_cols]
    categorical_ohe_cols = [f"{c}__ohe" for c in categorical_cols]

    if numeric_cols:
        stages.append(Imputer(inputCols=numeric_cols, outputCols=numeric_imputed_cols))

    if categorical_cols:
        stages.append(StringIndexer(
            inputCols=categorical_cols,
            outputCols=categorical_index_cols,
            handleInvalid="keep",
        ))
        stages.append(OneHotEncoder(
            inputCols=categorical_index_cols,
            outputCols=categorical_ohe_cols,
            handleInvalid="keep",
        ))

    stages.append(VectorAssembler(
        inputCols=numeric_imputed_cols + categorical_ohe_cols,
        outputCol="raw_features",
        handleInvalid="keep",
    ))

    if use_scaler:
        stages.append(StandardScaler(
            inputCol="raw_features",
            outputCol="features",
            withStd=True,
            withMean=False,
        ))
    else:
        stages.append(VectorAssembler(inputCols=["raw_features"], outputCol="features"))

    return stages


def evaluate_regression(predictions, model_name):
    rmse_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="rmse")
    mae_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="mae")
    r2_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="r2")

    rmse = float(rmse_eval.evaluate(predictions))
    mae = float(mae_eval.evaluate(predictions))
    r2 = float(r2_eval.evaluate(predictions))

    extra = predictions.select(
        avg(spark_abs(col(TARGET_COL) - col("prediction"))).alias("mae_check"),
        avg(spark_abs((col(TARGET_COL) - col("prediction")) / col(TARGET_COL))).alias("mape"),
    ).collect()[0]

    return {
        "model": model_name,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "mape": float(extra["mape"]),
    }


def train_regression_model(model_name, estimator, train_df, test_df, numeric_cols, categorical_cols, use_scaler=False):
    stages = build_preprocessing_stages(numeric_cols, categorical_cols, use_scaler=use_scaler)
    pipeline = Pipeline(stages=stages + [estimator])
    model = pipeline.fit(train_df)
    predictions = model.transform(test_df).persist(StorageLevel.MEMORY_AND_DISK)
    predictions.count()
    metrics = evaluate_regression(predictions, model_name)
    return model, predictions, metrics


def baseline_mean_model(train_df, test_df):
    mean_value = float(train_df.select(avg(col(TARGET_COL)).alias("mean_apr")).collect()[0]["mean_apr"])
    predictions = test_df.withColumn("prediction", col(TARGET_COL) * 0 + mean_value)
    metrics = evaluate_regression(predictions, "Baseline Mean Predictor")
    metrics["baseline_mean_prediction"] = mean_value
    return predictions, metrics


def plot_model_comparison(rows, output_path):
    try:
        import matplotlib.pyplot as plt
        import numpy as np
    except ImportError:
        print("matplotlib or numpy is not installed. Skip model comparison figure.")
        return

    models = [r["model"] for r in rows]
    rmse = [r["rmse"] for r in rows]
    mae = [r["mae"] for r in rows]
    r2 = [r["r2"] for r in rows]

    x = np.arange(len(models))
    width = 0.25

    fig, ax1 = plt.subplots(figsize=(12, 6))
    ax1.bar(x - width / 2, rmse, width, label="RMSE", color="#2F80ED")
    ax1.bar(x + width / 2, mae, width, label="MAE", color="#56CCF2")
    ax1.set_ylabel("Error")
    ax1.set_xticks(x)
    ax1.set_xticklabels(models, rotation=15, ha="right")
    ax1.legend(loc="upper left")

    ax2 = ax1.twinx()
    ax2.plot(x, r2, marker="o", linewidth=2, label="R2", color="#EB5757")
    ax2.set_ylabel("R2")
    ax2.legend(loc="upper right")

    plt.title("Regression Model Comparison")
    plt.tight_layout()
    plt.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.close()
    print(f"Saved figure: {output_path}")


def sample_predictions_to_pandas(predictions, sample_size=5000):
    return (
        predictions
        .select(TARGET_COL, "prediction")
        .orderBy(rand(seed=RANDOM_SEED))
        .limit(sample_size)
        .toPandas()
    )


def plot_actual_vs_predicted(predictions, output_path):
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib is not installed. Skip actual vs predicted figure.")
        return

    pdf = sample_predictions_to_pandas(predictions)
    plt.figure(figsize=(7, 7))
    plt.scatter(pdf[TARGET_COL], pdf["prediction"], alpha=0.35, color="#2F80ED", edgecolors="none")
    min_value = min(pdf[TARGET_COL].min(), pdf["prediction"].min())
    max_value = max(pdf[TARGET_COL].max(), pdf["prediction"].max())
    plt.plot([min_value, max_value], [min_value, max_value], linestyle="--", color="#EB5757")
    plt.xlabel("Actual BorrowerAPR")
    plt.ylabel("Predicted BorrowerAPR")
    plt.title("Actual vs Predicted BorrowerAPR")
    plt.tight_layout()
    plt.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.close()
    print(f"Saved figure: {output_path}")


def plot_residual_distribution(predictions, output_path):
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib is not installed. Skip residual distribution figure.")
        return

    pdf = (
        predictions
        .select((col(TARGET_COL) - col("prediction")).alias("residual"))
        .orderBy(rand(seed=RANDOM_SEED))
        .limit(20000)
        .toPandas()
    )

    plt.figure(figsize=(10, 6))
    plt.hist(pdf["residual"], bins=50, color="#2F80ED", edgecolor="white")
    plt.axvline(0, linestyle="--", color="#EB5757")
    plt.xlabel("Residual: actual - predicted")
    plt.ylabel("Frequency")
    plt.title("Residual Distribution for Best Regression Model")
    plt.tight_layout()
    plt.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.close()
    print(f"Saved figure: {output_path}")


def get_feature_names_from_pipeline_model(pipeline_model, numeric_cols, categorical_cols):
    names = list(numeric_cols)

    for stage in pipeline_model.stages:
        if stage.__class__.__name__ == "StringIndexerModel":
            for col_name, labels in zip(categorical_cols, stage.labelsArray):
                for label in labels:
                    names.append(f"{col_name}={label}")

    return names


def extract_tree_feature_importance(pipeline_model, numeric_cols, categorical_cols):
    final_model = pipeline_model.stages[-1]
    importances = list(final_model.featureImportances.toArray())
    names = get_feature_names_from_pipeline_model(pipeline_model, numeric_cols, categorical_cols)

    rows = []
    for i, importance in enumerate(importances):
        feature = names[i] if i < len(names) else f"feature_{i}"
        rows.append({"feature": feature, "importance": float(importance)})

    return sorted(rows, key=lambda row: row["importance"], reverse=True)


def plot_feature_importance(rows, output_path, top_n=15):
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib is not installed. Skip feature importance figure.")
        return

    top_rows = rows[:top_n]
    features = [r["feature"] for r in top_rows][::-1]
    scores = [r["importance"] for r in top_rows][::-1]

    plt.figure(figsize=(11, 7))
    plt.barh(features, scores, color="#2F80ED")
    plt.xlabel("Feature importance")
    plt.ylabel("Feature")
    plt.title(f"Top {top_n} Feature Importances - Best Regression Model")
    plt.tight_layout()
    plt.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.close()
    print(f"Saved figure: {output_path}")


def save_prediction_sample(predictions, path, sample_size=200):
    rows = []
    sample = (
        predictions
        .select(TARGET_COL, "prediction")
        .withColumn("absolute_error", spark_abs(col(TARGET_COL) - col("prediction")))
        .orderBy(rand(seed=RANDOM_SEED))
        .limit(sample_size)
        .collect()
    )
    for row in sample:
        rows.append({
            "actual_borrower_apr": float(row[TARGET_COL]),
            "predicted_borrower_apr": float(row["prediction"]),
            "absolute_error": float(row["absolute_error"]),
        })
    write_dicts_to_csv(rows, path)


def main():
    spark = SparkSession.builder.appName("Prosper Loan Spark ML Regression").getOrCreate()
    spark.sparkContext.setLogLevel("WARN")

    print_header("LOAD REGRESSION DATASET")
    df = spark.read.parquet(INPUT_PATH).persist(StorageLevel.MEMORY_AND_DISK)
    df.count()

    if TARGET_COL not in df.columns:
        raise ValueError(f"{TARGET_COL} is required for regression.")

    df = df.filter(col(TARGET_COL).isNotNull()).persist(StorageLevel.MEMORY_AND_DISK)
    summary = summarize_dataset(df)
    print(f"Rows: {summary['rows']}, Columns: {summary['columns']}")
    print(f"BorrowerAPR mean: {summary['target_mean']:.6f}")

    numeric_cols, categorical_cols = detect_feature_columns(df)
    print(f"Numeric features: {len(numeric_cols)}")
    print(f"Categorical features: {len(categorical_cols)}")
    print("BorrowerAPR is excluded from features because it is the regression target.")

    plot_target_distribution(df, TARGET_DISTRIBUTION_FIG)

    train_df, test_df = train_test_split(df)
    train_count = train_df.count()
    test_count = test_df.count()
    print(f"Train rows: {train_count}")
    print(f"Test rows : {test_count}")

    print_header("REGRESSION FEATURE DIAGNOSTIC")
    feature_rows = regression_feature_diagnostic(train_df, numeric_cols, categorical_cols)
    plot_feature_scores(feature_rows, FEATURE_SCORE_FIG)

    print_header("TRAIN REGRESSION MODELS")
    _, baseline_metrics = baseline_mean_model(train_df, test_df)

    lr = LinearRegression(
        labelCol=TARGET_COL,
        featuresCol="features",
        maxIter=80,
        regParam=0.01,
        elasticNetParam=0.0,
    )

    rf = RandomForestRegressor(
        labelCol=TARGET_COL,
        featuresCol="features",
        numTrees=100,
        maxDepth=8,
        seed=RANDOM_SEED,
    )

    gbt = GBTRegressor(
        labelCol=TARGET_COL,
        featuresCol="features",
        maxIter=80,
        maxDepth=5,
        stepSize=0.05,
        seed=RANDOM_SEED,
    )

    model_objects = {}
    prediction_objects = {}
    metric_rows = [baseline_metrics]

    for name, estimator, use_scaler in [
        ("Linear Regression", lr, True),
        ("Random Forest Regressor", rf, False),
        ("GBTRegressor", gbt, False),
    ]:
        model, predictions, metrics = train_regression_model(
            model_name=name,
            estimator=estimator,
            train_df=train_df,
            test_df=test_df,
            numeric_cols=numeric_cols,
            categorical_cols=categorical_cols,
            use_scaler=use_scaler,
        )
        model_objects[name] = model
        prediction_objects[name] = predictions
        metric_rows.append(metrics)
        print(f"{name}: RMSE={metrics['rmse']:.6f}, MAE={metrics['mae']:.6f}, R2={metrics['r2']:.6f}, MAPE={metrics['mape']:.6f}")

    write_dicts_to_csv(metric_rows, MODEL_METRICS_CSV)
    plot_model_comparison(metric_rows, MODEL_COMPARISON_FIG)

    candidate_rows = [row for row in metric_rows if row["model"] != "Baseline Mean Predictor"]
    best_row = sorted(candidate_rows, key=lambda row: (row["rmse"], row["mae"], -row["r2"]))[0]
    best_model_name = best_row["model"]
    best_model = model_objects[best_model_name]
    best_predictions = prediction_objects[best_model_name]

    print_header("BEST REGRESSION MODEL")
    print(f"Best model: {best_model_name}")
    print(f"RMSE={best_row['rmse']:.6f}, MAE={best_row['mae']:.6f}, R2={best_row['r2']:.6f}, MAPE={best_row['mape']:.6f}")

    plot_actual_vs_predicted(best_predictions, ACTUAL_VS_PREDICTED_FIG)
    plot_residual_distribution(best_predictions, RESIDUAL_DISTRIBUTION_FIG)
    save_prediction_sample(best_predictions, PREDICTION_SAMPLE_CSV)

    if best_model_name in ["Random Forest Regressor", "GBTRegressor"]:
        importance_rows = extract_tree_feature_importance(best_model, numeric_cols, categorical_cols)
        write_dicts_to_csv(importance_rows, FEATURE_IMPORTANCE_CSV)
        plot_feature_importance(importance_rows, FEATURE_IMPORTANCE_FIG)

    best_model.write().overwrite().save(MODEL_OUTPUT_PATH)
    print(f"Best model saved to: {MODEL_OUTPUT_PATH}")

    spark.stop()


if __name__ == "__main__":
    main()